# FEMTO / PRONOSTIA — HTT → LACA → Mamba fusion of vibration + temperature (A/B/C/D ablation + F0 + PROB-AVG)

**What this notebook tests.** It tests the originally proposed architecture on FEMTO's vibration + temperature
pair: modality tokens → **HTT** (Heterogeneous Temporal Tokenization) → **LACA** (Lag-Adaptive Cross-modal
Alignment) → **Mamba fusion** → classifier. The task is the same quartile degradation-stage task, under the same
leave-one-operating-condition-out protocol, as `femto-baselines.ipynb`. The labels, folds, inner-validation bearings,
seeds, early stopping and metrics are identical, so every number here can be compared directly with
`femto_baseline_results.csv`.

**Scale of the tokens (the central design decision).** HTT and LACA work at the **burst-token / trajectory scale**.
One vibration burst (2560 samples, 0.1 s) is **one token**. One temperature record is **one token**. A token
sequence spans **real elapsed time** across a `WINDOW_MINUTES`-long stretch of the bearing's trajectory. The only
physical reason to expect a cross-modal lag on this dataset is slow: temperature rise is a thermal consequence of
mechanical degradation and should trail vibration by minutes. So the tokens must carry timestamps at that scale.
Testing LACA within one 0.1 s burst would repeat the TII scale mismatch instead of giving the mechanism a fair test.

**Ablation (mandatory, TII 2×2 discipline).** All four conditions share the same encoders, Mamba, initial weights
(same seed → same construction order → same RNG draws) and training procedure. Only two switches differ:

| | HTT time embedding φ(τ) | LACA lag bias β·B(Δτ) |
|---|---|---|
| **A** | ON | OFF (β = 0, content-only attention) |
| **B** | OFF | ON |
| **C** | ON | ON (full proposed mechanism) |
| **D** | OFF | OFF (plain content-only cross-attention, no time information anywhere) |
| **F0** | OFF | no cross-attention at all (STEP 5 skipped): D's tokens concatenated straight into Mamba |
| **PROB-AVG** | — | analysis only, not a trained model (see its own section) |

**Dataset:** *FEMTO Bearing Data Set, NASA Prognostics Data Repository, NASA Ames Research Center.* P. Nectoux et
al., "PRONOSTIA: An experimental platform for bearings accelerated degradation tests", IEEE PHM 2012 Prognostic
Challenge. License: CC BY 3.0 US (citation required). The archive ships no license file.

**Self-sufficient, single Run All** on Kaggle (GPU, internet on). The one optional external input is the baseline
notebook's saved predictions, used only by PROB-AVG. If they are not attached, PROB-AVG is skipped with a message and
everything else still runs.

## Decisions stated before any result

**1. Bearings: 13 of 13, not 12.** The fusion set is every bearing that has *both* modalities. It is taken from the
baseline notebook's own constant, `EXPECTED_NO_TEMP`, in the ported section 1; it is not re-derived. That list is
`{Bearing1_3, Bearing2_3, Bearing2_2, Bearing3_2}`. The original audit also listed Bearing2_6, but the baseline run
found 116 real temperature files there (acc:temp 6.04, 600 rows per file). So **17 − 4 = 13 bearings** have both
modalities, and every fusion number is reported as "of 13". Section 8 (ported) still stops if the observed list ever
differs.

**2. Timestamp anchor (STEP 2) — `ANCHOR_MODE`.** Every token's time is elapsed time since **that bearing's first
vibration burst** (t = 0), so both modalities share one anchor. The per-bearing measured start offset
`temp_first − vib_first` is printed in a table with the 1–107 s range check. How the offset enters depends on
`ANCHOR_MODE`:
- **`"shared_clock"` (default).** Both CSV streams carry `hh:mm:ss` from the same acquisition clock. A temperature
  record stamped 107 s after the first burst really was taken 107 s later, so its time is `temp_raw − vib_first`.
  This is "temp elapsed since its own start + the measured offset", which puts both modalities on the same real
  clock. Subtracting the offset here would shift every temperature token earlier by 1–107 s and create a
  misalignment.
- **`"subtract_offset"`.** This is the literal STEP 2 instruction: `temp_raw − offset − vib_first`. It makes the
  first temperature record t = 0. It is correct **only if** the temperature logger's clock is skewed by the offset,
  i.e. both streams physically started together.

The data can tell these apart. Both streams stop when the test stops, so the **end gap** `temp_end − vib_end` should
be ≈ 0 under a shared clock and ≈ the start offset under clock skew. The anchor cell prints the end gap and the
evidence for each bearing. If the evidence contradicts the chosen mode, it warns loudly. `ANCHOR_MODE` is part of
the config tag, so switching it re-runs everything.

**3. Token features (STEP 1).**
- *Vibration:* per channel (horizontal, vertical), RMS, kurtosis (Pearson, normal = 3), peak |x| and crest factor
  (peak / RMS). That gives 8 features per token, the standard degradation indicators (Lei et al., 2018, *Machinery
  health prognostics: A systematic review from data acquisition to RUL prediction*). All 8 are positive and
  heavy-tailed near failure, so they are **log-transformed** before the per-fold z-score (`VIB_FEATURE_LOG = True`).
  This is a transformation, not a different feature set, and it can be switched off.
- *Temperature:* mean (°C) and least-squares slope (°C/s) of the record's readings, using row times `k / 10 Hz`.
  That gives 2 features. Records with fewer than `MIN_TEMP_ROWS` rows are skipped as tokens; the count is printed.
- A token's time is its file's first-row timestamp. The ported section 8 clock repair has already fixed Bearing1_1's
  two corrupted vibration timestamps.

**4. Windows and labels (STEP 3).** A window of `WINDOW_MINUTES` (10) slides with stride `WINDOW_STRIDE_MINUTES` (5)
over the bearing's vibration-defined life [0, L]. The last window is right-aligned to L, so the end of life is always
covered. A window holds every token of either modality whose time falls inside it. The number of tokens per window
varies, and batches are padded with masks. The label is the **baseline's own `stage_labels` function** evaluated at
the window's centre on the bearing's vibration timeline (the same lifetime definition as the baseline's vibration
labels). Only the baseline's exact formula is used, never a re-implementation.

**5. Model details kept from the prior TII code.**
- φ is the prior `ContinuousTimeEmbedding`: sinusoids of the continuous τ at 64 fixed frequencies, then a 2-layer
  MLP. τ is fed in **minutes** since the window start.
- HTT applies a LayerNorm to `Z̄ + e_m + φ(τ)`, as in the prior HTT.
- The lag bias g(Δτ) is a 1→32→1 MLP, zero-initialized, with a per-window grid LayerNorm (v18 stabilization).
  Here the grid norm is masked, so padding never enters it.
- β = 1 when ON and exactly 0 when OFF. γ starts at 0.5.
- All modules are built in **every** condition, so A/B/C/D draw identical initial weights at a given seed. A switch
  only decides whether a module is *used*. "Active" parameter counts subtract the unused ones.

The loss is class-weighted cross-entropy only, as in the baseline. The prior TII InfoNCE / consistency auxiliary
losses are left out to keep the training procedure identical to the baseline.

**6. Pre-registered verdict rule (fixed before any result).** LACA's lag bias shows a **reliable benefit** only if
**both** B − D and C − A pass all three tests, using test macro-F1 paired by (test condition, seed):
- mean difference > 0;
- positive in ≥ 5 of the 6 pairs;
- paired t-test p < 0.05.

Anything else is reported as **no reliable benefit**. With 3 folds × 2 seeds, this is a search-phase sample. A "no"
means the benefit was not shown, not that it was disproven.

## Ported baseline infrastructure (verbatim)

The cells below up to section 10, plus section 12, are **byte-for-byte copies** of `FEMTO/femto-baselines.ipynb`.
They cover recursive extraction, delimiter auto-detection, Test_set exclusion, the zero-temperature list, the clock
repair, `stage_labels`, the folds with bearing-level inner validation, `compute_metrics`, and all verification hooks.
The fusion constants cell sits between ported sections 1 and 2 and only *adds* names. Section 12 is ported only for
`compute_metrics`, `STAGES`, `STAGE_NAMES` and `run_tag` / `probs_path` (to find the baseline's prediction files). Its
baseline training functions are defined but never called. SHA-1 of each ported cell (checked again at runtime if the
baseline notebook file is available):

- `# 1. SETUP + CONSTANTS — every fixed value lives here` → `0677d0d6ea2e`
- `# 2. REAL mamba-ssm — install if missing, import, raise on failure (no fallback)` → `ee529627d6c7`
- `# 3. SEEDING, DETERMINISM FLAGS, HOOK HELPER` → `25054e2867c7`
- `## Data acquisition: download → recursive extraction → split discovery` → `0fe9801343b0`
- `# 4. DOWNLOAD (PHM Society mirror of the NASA Prognostics Data Repository)` → `af34f325387d`
- `# 5. RECURSIVE EXTRACTION — extract every .zip found, in passes, until none remain` → `65e2d7eb232c`
- `# 6. SPLIT DISCOVERY — Learning_set + Full_Test_Set only; Test_set located and EXCLUDED` → `ccbd0f1ecece`
- `## Parsing: per-file delimiter auto-detection, shape/NaN checks, timestamps` → `4b7643e06c71`
- `# 7. RAW CSV PARSING (guarded against Test_set, delimiter auto-detected per file, cached)` → `c56c60eb54c3`
- `# 8. INVENTORY + VERIFICATION: delimiters, shape/NaN, monotonic time, acc:temp ratio, zero-temperature bearings` → `7f8813beb204`
- `## Windows and degradation-stage proxy labels` → `468372ad0bab`
- `# 9. WINDOWS + PER-BEARING STAGE LABELS (+ optional vibration downsampling)` → `fdad19fb4b36`
- `# 10. LEAVE-ONE-CONDITION-OUT FOLDS + BEARING-LEVEL INNER VALIDATION` → `c387f26f37d7`
- `# 12. TRAINING / EVALUATION UTILITIES, results CSV (resumable)` → `5a9dbd3c3d8a`

In [ ]:
# ==============================================================================
# 1. SETUP + CONSTANTS — every fixed value lives here
# ==============================================================================
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # deterministic cuBLAS; must be set before CUDA initializes
import sys, re, json, time, math, copy, random, shutil, zipfile, hashlib, subprocess, urllib.request
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd

NOTEBOOK_T0 = time.time()

# --- paths: results/probs/checkpoints -> /kaggle/working (persisted); download/extraction/parse cache -> scratch ---
ON_KAGGLE = Path("/kaggle/working").is_dir()
WORK_DIR = Path("/kaggle/working") if ON_KAGGLE else Path.cwd() / "femto_working"
SCRATCH_DIR = Path("/tmp/femto") if ON_KAGGLE else WORK_DIR / "_scratch"
DOWNLOAD_PATH = SCRATCH_DIR / "download" / "femto_bearing.zip"
EXTRACT_DIR = SCRATCH_DIR / "extracted"
CACHE_DIR = SCRATCH_DIR / "parsed"
RESULTS_CSV = WORK_DIR / "femto_baseline_results.csv"
SUMMARY_CSV = WORK_DIR / "femto_baseline_summary.csv"
PROBS_DIR = WORK_DIR / "femto_probs"
CKPT_DIR = WORK_DIR / "femto_checkpoints"
for _d in (WORK_DIR, SCRATCH_DIR, CACHE_DIR, PROBS_DIR, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- download / extraction ---
FEMTO_URL = "https://phm-datasets.s3.amazonaws.com/NASA/10.+FEMTO+Bearing.zip"   # PHM Society mirror (NOT nasa.gov)
EXPECTED_ZIP_MB = 1157
MAX_EXTRACTION_PASSES = 10

# --- splits / bearings (quirks 3, 4, 7) ---
USE_SPLITS = ["Learning_set", "Full_Test_Set"]
EXCLUDED_SPLIT = "Test_set"                   # truncated copy of Full_Test_Set; never read
EXPECTED_SPLIT_BEARINGS = {
    "Learning_set": {"Bearing1_1", "Bearing1_2", "Bearing2_1", "Bearing2_2", "Bearing3_1", "Bearing3_2"},
    "Full_Test_Set": {"Bearing1_3", "Bearing1_4", "Bearing1_5", "Bearing1_6", "Bearing1_7",
                      "Bearing2_3", "Bearing2_4", "Bearing2_5", "Bearing2_6", "Bearing2_7", "Bearing3_3"},
}
EXPECTED_BEARINGS = set().union(*EXPECTED_SPLIT_BEARINGS.values())
# Audit listed Bearing2_6 too, but this notebook's sweep finds 116 real temp files there (ratio 6.04, 600 rows/file).
EXPECTED_NO_TEMP = {"Bearing1_3", "Bearing2_3", "Bearing2_2", "Bearing3_2"}
ALLOW_NO_TEMP_MISMATCH = False                # True = continue (loudly) if the zero-temperature list differs
CONDITIONS = {1: "~1800 rpm / 4.0 kN", 2: "~1650 rpm / 4.2 kN", 3: "~1500 rpm / 5.0 kN"}

# --- file formats (quirks 5, 6) ---
ACC_NCOLS, ACC_ROWS, ACC_FS = 6, 2560, 25_600     # hour, minute, second, microsecond, horiz_g, vert_g
ACC_SIGNAL_COLS = [4, 5]
ACC_DELIMS = (",", ";")                          # try comma first, then semicolon (auto-detected PER FILE)
TEMP_NCOLS, TEMP_FS = 5, 10                      # hour; minute; second; subsecond_index (0-9); temperature_C
TEMP_SIGNAL_COL = 4
TEMP_DELIMS = (";", ",")                         # documented semicolon; auto-detected per file all the same
ACC_PERIOD_S, TEMP_PERIOD_S = 10, 60             # documented acquisition periods -> expected acc:temp ≈ 6:1
RATIO_TOL = (5.0, 7.0)

# --- task / windows ---
N_STAGES = 4                                     # degradation-stage proxy bins (equal-width in elapsed time, per bearing)
VIB_DOWNSAMPLE_FACTOR = 1                        # OFF by default; >1 applies scipy.signal.decimate (anti-aliased)

# --- run matrix ---
SEEDS = [42, 123]
INNER_VAL_SEED = 2026                            # bearing-level inner-validation choice (fixed per fold)
FOLD_CONDITIONS = [1, 2, 3]                      # the held-out (test) condition of each fold
RUNS = [("vibration", "mamba"), ("vibration", "cnn"), ("temperature", "mamba")]

# --- (A) encoder + Mamba (prior UnimodalMamba pattern) ---
D_MODEL = 256
ENC_HIDDEN = 64
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
POOL_HIDDEN = 64
DROPOUT = 0.15

# --- (B) WDCNN-style 1D-CNN (vibration only) ---
CNN_K1, CNN_S1, CNN_C1 = 64, 16, 16              # wide first kernel (WDCNN: 64, stride 16, 16 filters)
CNN_POOL1 = 2
CNN_K2, CNN_C2 = 50, 32                          # ~50-sample second kernel

# --- optimization (prior work's recipe; CNN uses WDCNN's customary 1e-3 learning rate) ---
EPOCHS = 20
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
LEARNING_RATE = {"mamba": 1e-4, "cnn": 1e-3}
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6
GRAD_CLIP_NORM = 1.0
EARLY_STOP_PATIENCE = 5                          # stop after this many epochs without a new best val macro-F1
NUM_WORKERS = 2
NORM_EPS = 1e-6

# A results row / prediction file counts as "done" only under the SAME config tag (changing N_STAGES etc. re-runs).
CONFIG_TAG = f"S{N_STAGES}_ds{VIB_DOWNSAMPLE_FACTOR}_e{EPOCHS}_p{EARLY_STOP_PATIENCE}_v{INNER_VAL_SEED}"

print(f"Kaggle: {ON_KAGGLE} | outputs -> {WORK_DIR} | scratch -> {SCRATCH_DIR}")
print(f"N_STAGES {N_STAGES} | VIB_DOWNSAMPLE_FACTOR {VIB_DOWNSAMPLE_FACTOR} | seeds {SEEDS} | runs {RUNS} | "
      f"config tag {CONFIG_TAG}")

In [ ]:
# ==============================================================================
# F1. FUSION CONSTANTS + OUTPUT PATHS (adds names only; ported baseline constants above are unchanged)
# ==============================================================================
assert VIB_DOWNSAMPLE_FACTOR == 1, "vibration token features are defined on the raw 25.6 kHz bursts"
BASELINE_CONFIG_TAG = CONFIG_TAG              # the baseline's tag; used only to match its saved predictions (PROB-AVG)

FUSION_RESULTS_CSV = WORK_DIR / "femto_fusion_results.csv"
FUSION_SUMMARY_CSV = WORK_DIR / "femto_fusion_summary.csv"
FUSION_PAIRED_CSV = WORK_DIR / "femto_fusion_paired.csv"
FUSION_PROBS_DIR = WORK_DIR / "femto_fusion_probs"
FUSION_CKPT_DIR = WORK_DIR / "femto_fusion_checkpoints"
for _d in (FUSION_PROBS_DIR, FUSION_CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- STEP 1: per-burst token features ---
N_VIB_FEAT, N_TEMP_FEAT = 8, 2
VIB_FEATURE_LOG = True                        # log() of the 8 positive, heavy-tailed vibration features before z-score
MIN_TEMP_ROWS = 10                            # a temperature record needs >= this many rows to give a mean + slope
FEAT_EPS = 1e-12
FEAT_NORM_EPS = 1e-6

# --- STEP 2: timestamp anchor ---
ANCHOR_MODE = "shared_clock"                  # "shared_clock" (default, see markdown) or "subtract_offset" (literal)
OFFSET_EXPECTED_RANGE_S = (1.0, 107.0)        # prior audit's measured vib->temp start-offset range
OFFSET_FAR_MARGIN_S = 30.0                    # outside the range by more than this = "far" -> hard stop
ALLOW_FAR_OFFSETS = False                     # True = continue (loudly) past a far-outside offset
CLOCK_EVIDENCE_TOL_S = 15.0                   # end-gap tolerance for the shared-clock vs clock-skew diagnostic
assert ANCHOR_MODE in ("shared_clock", "subtract_offset")

# --- STEP 3: windows ---
WINDOW_MINUTES = 10
WINDOW_STRIDE_MINUTES = 5
MIN_TOKENS_PER_MODALITY = 1                   # a window needs >= 1 token of EACH modality
TAU_UNIT_S = 60.0                             # tau (HTT) and delta-tau (LACA) are fed to the model in minutes

# --- STEPS 4-6: HTT / LACA / Mamba (Mamba, D_MODEL, POOL_HIDDEN, DROPOUT reused from the baseline constants) ---
TIME_EMBED_FREQUENCIES = 64
LACA_NUM_HEADS = 4
LACA_BETA = 1.0                               # lag bias ON; OFF = exactly 0.0
LAG_BIAS_HIDDEN = 32
LAG_BIAS_PARAMS = 97                          # Linear(1,32) + Linear(32,1)
LAG_BIAS_LN_EPS = 1e-5
GAMMA_INIT = 0.5

# --- ablation conditions: (HTT time embedding ON?, LACA lag bias ON?, cross-attention present?) ---
FUSION_MODELS = ["A", "B", "C", "D", "F0"]
ABLATION = {"A": (True, False, True), "B": (False, True, True), "C": (True, True, True), "D": (False, False, True),
            "F0": (False, False, False)}

# --- training: identical to the baseline Mamba runs (EPOCHS, EARLY_STOP_PATIENCE, BATCH_SIZE, WEIGHT_DECAY, ...) ---
FUSION_LR = LEARNING_RATE["mamba"]
FUSION_NUM_WORKERS = 0                        # token tables are small; in-process loading keeps the RNG stream simple

# --- PROB-AVG: optional GitHub fallback for the baseline's saved probabilities ("" = off, no internet needed) ---
BASELINE_PROBS_GITHUB_URL = ""                # repo URL (git clone --depth 1, then find femto_probs / Baseline Probs),
                                              # a github.com/<o>/<r>/tree/<branch>/<folder> URL, or a raw/direct folder URL

# --- pre-registered verdict rule ---
VERDICT_MIN_WINS = 5                          # of 6 (fold, seed) pairs
VERDICT_ALPHA = 0.05

FUSION_CONFIG_TAG = (f"S{N_STAGES}_w{WINDOW_MINUTES}s{WINDOW_STRIDE_MINUTES}_{ANCHOR_MODE}_log{int(VIB_FEATURE_LOG)}"
                     f"_e{EPOCHS}_p{EARLY_STOP_PATIENCE}_v{INNER_VAL_SEED}")
print(f"Fusion models {FUSION_MODELS} | seeds {SEEDS} | window {WINDOW_MINUTES} min / stride {WINDOW_STRIDE_MINUTES} "
      f"min | anchor {ANCHOR_MODE} | tag {FUSION_CONFIG_TAG}")

In [ ]:
# ==============================================================================
# 2. REAL mamba-ssm — install if missing, import, raise on failure (no fallback)
# ==============================================================================
import importlib, importlib.util
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


def _pip(args):
    print("  pip install " + " ".join(args))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stderr or r.stdout)[-2000:])
    return r.returncode == 0


if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU: real mamba-ssm needs a GPU. Enable a GPU accelerator and re-run.")
if importlib.util.find_spec("mamba_ssm") is None:
    _ok = (_pip(["packaging", "ninja"]) and _pip(["causal-conv1d>=1.2.0", "--no-build-isolation"])
           and _pip(["mamba-ssm", "--no-build-isolation"]))
    importlib.invalidate_caches()
    if not _ok:
        raise RuntimeError("mamba-ssm installation FAILED (see pip output). Check internet + GPU, restart, re-run.")
try:
    import mamba_ssm
    from mamba_ssm import Mamba as MAMBA_CLASS
except Exception as exc:
    raise RuntimeError(f"`from mamba_ssm import Mamba` failed: {exc!r}. No fallback is used.") from exc
assert MAMBA_CLASS.__module__.startswith("mamba_ssm"), f"Mamba class comes from {MAMBA_CLASS.__module__}, not mamba_ssm"
device = torch.device("cuda")
print(f"mamba_ssm {getattr(mamba_ssm, '__version__', '?')} ({MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__}) | torch "
      f"{torch.__version__} | CUDA {torch.version.cuda} | GPU {torch.cuda.get_device_name(0)}")

In [ ]:
# ==============================================================================
# 3. SEEDING, DETERMINISM FLAGS, HOOK HELPER
# ==============================================================================
def seed_all(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_generator(seed):
    g = torch.Generator()
    g.manual_seed(seed)
    return g


def worker_init_fn(worker_id):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
seed_all(42)

HOOKS = defaultdict(list)


def hook(name, ok, msg, stop=True):
    ok = bool(ok)
    HOOKS[name] = [(ok, msg, stop)]                  # re-running a cell replaces that hook's earlier result
    print(f"[HOOK {name}] {'PASS' if ok else 'FAIL'} — {msg}")
    if not ok and stop:
        raise RuntimeError(f"HOOK {name} FAILED: {msg}")
    return ok


def loud(msg):
    print("🚨" * 50)
    for line in msg.split("\n"):
        print("🚨  " + line)
    print("🚨" * 50)

## Data acquisition: download → recursive extraction → split discovery

The outer zip holds `10. FEMTO Bearing/FEMTOBearingDataSet.zip`. That zip holds `Training_set.zip`, `Test_set.zip`
and `Validation_Set.zip`. Extraction is **recursive**: each pass finds every `.zip` left in the tree, extracts it in
place and removes it. Passes repeat until no `.zip` remains. The downloaded archive is kept outside the extraction
tree, so a rerun in the same session neither downloads nor extracts again.

In [ ]:
# ==============================================================================
# 4. DOWNLOAD (PHM Society mirror of the NASA Prognostics Data Repository)
# ==============================================================================
def download(url, dest, retries=3):
    if dest.exists() and zipfile.is_zipfile(dest):
        print(f"Archive already present (re-used): {dest} ({dest.stat().st_size / 1e6:,.0f} MB)")
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_name(dest.name + ".part")
    for attempt in range(1, retries + 1):
        try:
            print(f"Downloading {url} (attempt {attempt}/{retries}) ...")
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120) as r, open(part, "wb") as f:
                total = int(r.headers.get("Content-Length") or 0)
                done, next_report = 0, 0
                while True:
                    chunk = r.read(8 << 20)
                    if not chunk:
                        break
                    f.write(chunk)
                    done += len(chunk)
                    if done >= next_report:
                        print(f"  {done / 1e6:8,.0f} / {total / 1e6:,.0f} MB")
                        next_report += 100_000_000
            if total and done != total:
                raise IOError(f"truncated download: {done:,} of {total:,} bytes")
            part.replace(dest)
            return
        except Exception as exc:
            print(f"  attempt {attempt} failed: {exc!r}")
            if attempt == retries:
                raise RuntimeError(f"Download of {url} failed after {retries} attempts (is Kaggle internet ON?)") from exc
            time.sleep(15 * attempt)


_t = time.time()
download(FEMTO_URL, DOWNLOAD_PATH)
_size_mb, _size_mib = DOWNLOAD_PATH.stat().st_size / 1e6, DOWNLOAD_PATH.stat().st_size / 2 ** 20
hook("download_is_zip", zipfile.is_zipfile(DOWNLOAD_PATH), f"{DOWNLOAD_PATH.name} is a valid zip ({time.time() - _t:.0f} s)")
hook("download_size", min(abs(_size_mb - EXPECTED_ZIP_MB), abs(_size_mib - EXPECTED_ZIP_MB)) / EXPECTED_ZIP_MB < 0.10,
     f"{_size_mb:,.0f} MB ({_size_mib:,.0f} MiB) vs documented ~{EXPECTED_ZIP_MB} MB", stop=False)

In [ ]:
# ==============================================================================
# 5. RECURSIVE EXTRACTION — extract every .zip found, in passes, until none remain
# ==============================================================================
def is_junk(p):
    """macOS resource-fork artefacts (__MACOSX/, ._name) are not data and not real archives."""
    p = Path(p)
    return "__MACOSX" in p.parts or p.name.startswith("._")


def zips_in(root):
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() == ".zip")


SENTINEL = EXTRACT_DIR / ".femto_extraction_complete.json"
if SENTINEL.exists():
    EXTRACTION_LOG = json.loads(SENTINEL.read_text())
    print(f"Extraction already complete (sentinel present) — re-using {EXTRACT_DIR}")
else:
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)                       # a half-finished earlier extraction is redone from scratch
    EXTRACT_DIR.mkdir(parents=True)
    _t = time.time()
    with zipfile.ZipFile(DOWNLOAD_PATH) as z:
        z.extractall(EXTRACT_DIR)
        EXTRACTION_LOG = [{"pass": 1, "zip": DOWNLOAD_PATH.name, "members": len(z.namelist())}]
    n_pass = 1
    while True:
        found = zips_in(EXTRACT_DIR)
        for p in [p for p in found if is_junk(p)]:
            p.unlink()
        found = [p for p in found if not is_junk(p)]
        if not found:
            break
        n_pass += 1
        if n_pass > MAX_EXTRACTION_PASSES:
            raise RuntimeError(f"Still finding .zip files after {MAX_EXTRACTION_PASSES} passes: {found[:5]}")
        for zp in found:
            with zipfile.ZipFile(zp) as z:
                z.extractall(zp.parent)
                n_members = len(z.namelist())
            EXTRACTION_LOG.append({"pass": n_pass, "zip": str(zp.relative_to(EXTRACT_DIR)), "members": n_members})
            zp.unlink()                                  # expanded in place; removing it is what lets the loop end
    SENTINEL.write_text(json.dumps(EXTRACTION_LOG))
    print(f"Extraction finished in {time.time() - _t:.0f} s")

for e in EXTRACTION_LOG:
    print(f"  pass {e['pass']}: {e['zip']}  ({e['members']:,} members)")
_remaining = [p for p in zips_in(EXTRACT_DIR) if not is_junk(p)]
_n_passes = max(e["pass"] for e in EXTRACTION_LOG)
hook("recursive_extraction", not _remaining,
     f"{len(EXTRACTION_LOG)} archives extracted over {_n_passes} passes; .zip files remaining anywhere: {len(_remaining)}")
hook("nested_depth", _n_passes >= 3, f"{_n_passes} extraction passes (documented nesting needs >= 3)", stop=False)

In [ ]:
# ==============================================================================
# 6. SPLIT DISCOVERY — Learning_set + Full_Test_Set only; Test_set located and EXCLUDED
# ==============================================================================
BEARING_RE = re.compile(r"^Bearing(\d)_(\d+)$")


def bearing_children(d):
    return sorted(c for c in d.iterdir() if c.is_dir() and BEARING_RE.match(c.name))


ALL_DIRS = [Path(r) for r, _, _ in os.walk(EXTRACT_DIR) if not is_junk(r)]


def find_split(name):
    """Directories named `name` (case-insensitive) that directly contain Bearing*_* folders."""
    return [d for d in ALL_DIRS if d.name.lower() == name.lower() and bearing_children(d)]


SPLIT_ROOTS = {}
for _name in USE_SPLITS:
    _found = find_split(_name)
    hook(f"split_{_name}", len(_found) == 1,
         f"{_name}/ found exactly once: {[str(p.relative_to(EXTRACT_DIR)) for p in _found]}")
    SPLIT_ROOTS[_name] = _found[0]

# Test_set: only its folder NAMES are listed (to show it mirrors Full_Test_Set). No file under it is ever opened.
_excluded = find_split(EXCLUDED_SPLIT)
for _d in _excluded:
    print(f"EXCLUDED {_d.relative_to(EXTRACT_DIR)}/ (truncated PHM2012 copy) — bearing folders: "
          f"{[c.name for c in bearing_children(_d)]} — none of its files are read")
if not _excluded:
    print(f"Note: no {EXCLUDED_SPLIT}/ folder found (nothing to exclude).")

BEARINGS, _dups = {}, []
for _split, _root in SPLIT_ROOTS.items():
    for _d in bearing_children(_root):
        if _d.name in BEARINGS:
            _dups.append(_d.name)
        _m = BEARING_RE.match(_d.name)
        BEARINGS[_d.name] = {"split": _split, "dir": _d, "condition": int(_m.group(1)), "index": int(_m.group(2))}
BEARING_NAMES = sorted(BEARINGS, key=lambda b: (BEARINGS[b]["condition"], BEARINGS[b]["index"]))
COND_OF = {b: BEARINGS[b]["condition"] for b in BEARING_NAMES}

hook("17_bearings", len(BEARINGS) == 17 and set(BEARINGS) == EXPECTED_BEARINGS and not _dups,
     f"{len(BEARINGS)} distinct bearings across {USE_SPLITS} (duplicates across splits: {_dups}); "
     f"missing {sorted(EXPECTED_BEARINGS - set(BEARINGS))}, unexpected {sorted(set(BEARINGS) - EXPECTED_BEARINGS)}")
for _split, _exp in EXPECTED_SPLIT_BEARINGS.items():
    _got = {b for b in BEARINGS if BEARINGS[b]["split"] == _split}
    hook(f"split_members_{_split}", _got == _exp, f"{_split}: {len(_got)} bearings {sorted(_got)}")
hook("excluded_split_not_used", all(EXCLUDED_SPLIT.lower() not in [p.lower() for p in BEARINGS[b]["dir"].parts]
                                    for b in BEARINGS), f"no bearing directory lies under {EXCLUDED_SPLIT}/")
for _c, _desc in CONDITIONS.items():
    print(f"  condition {_c} ({_desc}): {[b for b in BEARING_NAMES if COND_OF[b] == _c]}")

## Parsing: per-file delimiter auto-detection, shape/NaN checks, timestamps

Every CSV is parsed with **per-file delimiter auto-detection**. Accelerometer files try comma first; if that does not
give 6 columns, they retry with semicolon. Temperature files try semicolon (documented) first, then comma, and need 5
columns. Nothing is special-cased by folder name. `Full_Test_Set/Bearing1_4`'s semicolon-delimited acc files are
expected to be caught by this rule, and any other such file is reported. Every acc file must have exactly 2560
finite rows. Every temp file is checked for NaN, and its row count is **measured** and printed rather than assumed.
A file's time is its first row's `hour*3600 + minute*60 + second + subsecond`. A midnight rollover, if any, is
unwrapped. **Clock glitches are repaired, never ignored.** When a file's time is ≤ the previous file's time, the
repair looks ahead (up to `MAX_EXCURSION_FILES` files) for the first file whose time is again later than the last
good one, i.e. back on the original timeline. The files in between have **corrupted timestamps**, and their times are
linearly interpolated between the two good neighbours. Every other timestamp, including genuine forward gaps, is left
as recorded. Example: Bearing1_1 acc files 2121–2122 both read 09:38:46 inside an otherwise continuous 15:32–15:33
run. If the clock never comes back (a persistent reset), that single step is set to the bearing's median step and the
rest of the timeline is shifted. Each repair is printed with file numbers, raw timestamps and the times it was given.
The stage labels of a repaired bearing are compared with file-order labels. More than `MAX_CLOCK_REPAIRS_PER_BEARING`
repair events on one bearing means it is not an isolated glitch, and the notebook stops. After repair, each bearing's
timeline must be strictly increasing. A **span check** (`recorded span ≈ (n_files − 1) × median period`) flags any
bearing whose timeline length disagrees with its file count. Parsed arrays are cached in scratch space so a
rerun in the same session skips re-parsing.

In [ ]:
# ==============================================================================
# 7. RAW CSV PARSING (guarded against Test_set, delimiter auto-detected per file, cached)
# ==============================================================================
PARSER_VERSION = 1
READ_COUNT = Counter()                       # files read, by split


def guard_not_excluded(path):
    if EXCLUDED_SPLIT.lower() in [p.lower() for p in Path(path).parts]:
        raise RuntimeError(f"Refusing to read {path}: it is under the excluded {EXCLUDED_SPLIT}/ split")


def read_csv_autodelim(path, expected_ncols, delimiters, split):
    """Try each delimiter in order; return the first parse with exactly `expected_ncols` columns."""
    guard_not_excluded(path)
    for sep in delimiters:
        df = pd.read_csv(path, header=None, sep=sep, engine="c")
        if df.shape[1] == expected_ncols:
            READ_COUNT[split] += 1
            return df.to_numpy(dtype=np.float64), sep
    raise ValueError(f"{path}: none of the delimiters {delimiters} gives {expected_ncols} columns")


def file_num(p):
    return int(re.search(r"_(\d+)\.csv$", p.name).group(1))


def parse_bearing(name):
    info = BEARINGS[name]
    cache = CACHE_DIR / f"{name}_p{PARSER_VERSION}.npz"
    if cache.exists():
        with np.load(cache, allow_pickle=False) as z:
            out = {k: z[k] for k in z.files}
        READ_COUNT[info["split"] + " (cached)"] += len(out["acc_t_raw"]) + len(out["temp_t_raw"])
        return out
    acc_files = sorted(info["dir"].glob("acc_*.csv"), key=file_num)
    temp_files = sorted(info["dir"].glob("temp_*.csv"), key=file_num)
    acc_x = np.full((len(acc_files), ACC_ROWS, len(ACC_SIGNAL_COLS)), np.nan, np.float32)
    acc_t, acc_sep, acc_bad = np.full(len(acc_files), np.nan), [], []
    for i, f in enumerate(acc_files):
        a, sep = read_csv_autodelim(f, ACC_NCOLS, ACC_DELIMS, info["split"])
        acc_sep.append(sep)
        n_bad = int((~np.isfinite(a)).sum())
        if a.shape[0] != ACC_ROWS or n_bad:
            acc_bad.append(f"{f.name}: rows {a.shape[0]} (expected {ACC_ROWS}), non-finite values {n_bad}")
            continue
        acc_x[i] = a[:, ACC_SIGNAL_COLS]
        acc_t[i] = a[0, 0] * 3600 + a[0, 1] * 60 + a[0, 2] + a[0, 3] * 1e-6
    temp_chunks, temp_len, temp_t, temp_sep, temp_bad = [], [], np.full(len(temp_files), np.nan), [], []
    for i, f in enumerate(temp_files):
        a, sep = read_csv_autodelim(f, TEMP_NCOLS, TEMP_DELIMS, info["split"])
        temp_sep.append(sep)
        n_bad = int((~np.isfinite(a)).sum())
        if n_bad or a.shape[0] == 0:
            temp_bad.append(f"{f.name}: rows {a.shape[0]}, non-finite values {n_bad}")
        temp_chunks.append(a[:, TEMP_SIGNAL_COL].astype(np.float32))
        temp_len.append(a.shape[0])
        if a.shape[0]:
            temp_t[i] = a[0, 0] * 3600 + a[0, 1] * 60 + a[0, 2] + a[0, 3] / TEMP_FS
    out = {
        "acc_x": acc_x, "acc_t_raw": acc_t, "acc_num": np.array([file_num(f) for f in acc_files], np.int64),
        "acc_sep": np.array(acc_sep, dtype="<U1"), "acc_bad": np.array(acc_bad, dtype=str),
        "temp_flat": np.concatenate(temp_chunks) if temp_chunks else np.zeros(0, np.float32),
        "temp_len": np.array(temp_len, np.int64), "temp_t_raw": temp_t,
        "temp_num": np.array([file_num(f) for f in temp_files], np.int64),
        "temp_sep": np.array(temp_sep, dtype="<U1"), "temp_bad": np.array(temp_bad, dtype=str),
    }
    np.savez(cache, **out)
    return out


DATA = {}
_t = time.time()
for _b in BEARING_NAMES:
    _tb = time.time()
    DATA[_b] = parse_bearing(_b)
    print(f"  {BEARINGS[_b]['split']:>13}/{_b:<11} acc files {len(DATA[_b]['acc_t_raw']):5d} | temp files "
          f"{len(DATA[_b]['temp_t_raw']):4d} | {time.time() - _tb:5.1f} s")
print(f"Parsed 17 bearings in {time.time() - _t:.0f} s | files read by split: {dict(READ_COUNT)}")
hook("test_set_never_read", all(EXCLUDED_SPLIT.lower() != k.split(" ")[0].lower() for k in READ_COUNT)
     and set(k.split(" ")[0] for k in READ_COUNT) <= set(USE_SPLITS),
     f"files were read only from {sorted(set(k.split(' ')[0] for k in READ_COUNT))}; {EXCLUDED_SPLIT}: 0 files")

In [ ]:
# ==============================================================================
# 8. INVENTORY + VERIFICATION: delimiters, shape/NaN, monotonic time, acc:temp ratio, zero-temperature bearings
# ==============================================================================
MAX_CLOCK_REPAIRS_PER_BEARING = 5                    # more repair events than this on one bearing -> stop
MAX_EXCURSION_FILES = 10                             # look-ahead for the clock to return to the original timeline
SPAN_TOL = 0.02                                      # |span / ((n-1) x median period) - 1| allowed by the span check


def unwrap_midnight(t):
    jumps = np.diff(t) < -43_200                     # clock went back > 12 h: a midnight rollover
    return t + np.concatenate([[0.0], np.cumsum(jumps)]) * 86_400, int(jumps.sum())


def hms(sec):
    return f"{int(sec // 3600):02d}:{int(sec % 3600 // 60):02d}:{sec % 60:09.6f}"


def repair_clock(t_raw, nums, label):
    """(1) Unwrap midnight rollovers. (2) At each non-positive step after file i, find the first later file j (within
    MAX_EXCURSION_FILES) whose time is > t[i]: files i+1..j-1 carry corrupted timestamps and are linearly
    interpolated between t[i] and t[j]. If the clock never returns, treat it as a persistent reset: set that one step
    to the median step and shift the rest of the timeline. All other timestamps are left as recorded."""
    if len(t_raw) < 2 or not np.all(np.isfinite(t_raw)):
        return t_raw.copy(), 0, []                   # non-finite times are left for the shape_nan / monotonic hooks
    t, wraps = unwrap_midnight(t_raw)
    t = t.copy()
    d = np.diff(t)
    med = float(np.median(d[d > 0]))
    repairs, i = [], 0
    while i < len(t) - 1:
        if t[i + 1] > t[i]:
            i += 1
            continue
        ahead = np.where(t[i + 1:i + 2 + MAX_EXCURSION_FILES] > t[i])[0]
        if len(ahead):
            j = i + 1 + int(ahead[0])
            new = np.interp(np.arange(i + 1, j), [i, j], [t[i], t[j]])
            repairs.append(
                f"{label}: CORRUPTED TIMESTAMPS on {j - i - 1} file(s) {nums[i + 1:j].tolist()} between good files "
                f"{nums[i]} ({hms(t_raw[i])}) and {nums[j]} ({hms(t_raw[j])}): raw "
                f"{[hms(x) for x in t_raw[i + 1:j]]} -> interpolated {[hms(x % 86_400) for x in new]} "
                f"(step {(t[j] - t[i]) / (j - i):.3f} s; bearing median {med:.3f} s)")
            t[i + 1:j] = new
            i = j
        else:
            shift = t[i] + med - t[i + 1]
            repairs.append(
                f"{label}: PERSISTENT CLOCK RESET after file {nums[i]}: raw {hms(t_raw[i])} -> {hms(t_raw[i + 1])} "
                f"(step {t[i + 1] - t[i]:+.3f} s, no return within {MAX_EXCURSION_FILES} files) -> step set to median "
                f"{med:.3f} s, the remaining {len(t) - i - 1} files shifted by {shift:+.3f} s")
            t[i + 1:] += shift
            i += 1
    return t, wraps, repairs


_rows, _bad_files, _nonmono = [], [], []
CLOCK_REPAIRS = {}
for _b in BEARING_NAMES:
    D = DATA[_b]
    D["acc_t"], _aw, _ar = repair_clock(D["acc_t_raw"], D["acc_num"], f"{_b} acc")
    D["temp_t"], _tw, _tr = repair_clock(D["temp_t_raw"], D["temp_num"], f"{_b} temp")
    if _ar or _tr:
        CLOCK_REPAIRS[_b] = {"acc": _ar, "temp": _tr}
    for _arr, _kind in ((D["acc_t"], "acc"), (D["temp_t"], "temp")):
        if len(_arr) > 1 and not (np.all(np.isfinite(_arr)) and np.all(np.diff(_arr) > 0)):
            _nonmono.append(f"{_b} {_kind}")
    _bad_files += [f"{_b}/{s}" for s in list(D["acc_bad"]) + list(D["temp_bad"])]
    _n_acc, _n_temp = len(D["acc_t"]), len(D["temp_t"])
    _rows.append({
        "bearing": _b, "split": BEARINGS[_b]["split"], "cond": COND_OF[_b],
        "n_acc": _n_acc, "n_temp": _n_temp,
        "acc_comma": int((D["acc_sep"] == ",").sum()), "acc_semicolon": int((D["acc_sep"] == ";").sum()),
        "temp_semicolon": int((D["temp_sep"] == ";").sum()), "temp_comma": int((D["temp_sep"] == ",").sum()),
        "temp_rows": ",".join(str(v) for v in sorted(set(D["temp_len"].tolist()))) or "-",
        "acc:temp": round(_n_acc / _n_temp, 2) if _n_temp else np.nan,
        "acc_period_s": round(float(np.median(np.diff(D["acc_t"]))), 2) if _n_acc > 1 else np.nan,
        "temp_period_s": round(float(np.median(np.diff(D["temp_t"]))), 2) if _n_temp > 1 else np.nan,
        "life_h": round((D["acc_t"][-1] - D["acc_t"][0]) / 3600, 2),
        "midnight_wraps": _aw + _tw,
        "clock_repairs": len(_ar) + len(_tr),
        "temp_start_minus_acc_start_s (info only)": round(D["temp_t"][0] - D["acc_t"][0], 1) if _n_temp else np.nan,
    })
INVENTORY = pd.DataFrame(_rows)
with pd.option_context("display.width", 250, "display.max_columns", 30):
    print(INVENTORY.to_string(index=False))
print(f"Totals: {INVENTORY.n_acc.sum():,} acc files, {INVENTORY.n_temp.sum():,} temp files")

# --- shape / NaN safety net, monotonic timestamps ---
hook("shape_nan", not _bad_files, f"every acc file has {ACC_ROWS} finite rows and every temp file is finite; "
     f"violations: {len(_bad_files)} {_bad_files[:10]}")
if CLOCK_REPAIRS:
    loud("CLOCK DISCONTINUITIES REPAIRED (non-positive step between consecutive files):\n" + "\n".join(
        r for v in CLOCK_REPAIRS.values() for r in v["acc"] + v["temp"]))
_too_many = {b: len(v["acc"]) + len(v["temp"]) for b, v in CLOCK_REPAIRS.items()
             if len(v["acc"]) > MAX_CLOCK_REPAIRS_PER_BEARING or len(v["temp"]) > MAX_CLOCK_REPAIRS_PER_BEARING}
hook("clock_repairs_isolated", not _too_many,
     f"clock repairs per bearing: { {b: len(v['acc']) + len(v['temp']) for b, v in CLOCK_REPAIRS.items()} or 'none'} "
     f"(limit {MAX_CLOCK_REPAIRS_PER_BEARING} per modality per bearing; over the limit: {_too_many or 'none'})")
hook("monotonic_time", not _nonmono, f"per-bearing timestamps strictly increasing after midnight unwrap + clock "
     f"repair; violations: {_nonmono}; midnight wraps unwrapped: {int(INVENTORY.midnight_wraps.sum())}; isolated "
     f"clock repairs: {int(INVENTORY.clock_repairs.sum())}")

# --- span check: each modality's recorded span should match its file count x median period ---
_span_bad = []
for _b in BEARING_NAMES:
    for _key in ("acc_t", "temp_t"):
        _t = DATA[_b][_key]
        if len(_t) > 2:
            _ratio = (_t[-1] - _t[0]) / ((len(_t) - 1) * np.median(np.diff(_t)))
            if abs(_ratio - 1) > SPAN_TOL:
                _span_bad.append(f"{_b} {_key[:-2]}: span / ((n-1) x median period) = {_ratio:.3f}")
if _span_bad:
    loud("RECORDED SPAN DISAGREES WITH FILE COUNT (a gap or an unrepaired clock jump):\n" + "\n".join(_span_bad))
hook("span_vs_file_count", not _span_bad, f"every bearing's span is within {SPAN_TOL:.0%} of (n_files - 1) x median "
     f"period; exceptions: {_span_bad or 'none'}", stop=False)

# --- delimiter auto-detection ---
_semi_acc = INVENTORY[INVENTORY.acc_semicolon > 0]
_comma_temp = INVENTORY[INVENTORY.temp_comma > 0]
print("\nDelimiter auto-detection:")
print(f"  acc files: {int(INVENTORY.acc_comma.sum()):,} comma, {int(INVENTORY.acc_semicolon.sum()):,} semicolon")
for r in _semi_acc.itertuples():
    print(f"    semicolon acc files in {r.split}/{r.bearing}: {r.acc_semicolon} of {r.n_acc}")
print(f"  temp files: {int(INVENTORY.temp_semicolon.sum()):,} semicolon, {int(INVENTORY.temp_comma.sum()):,} comma")
for r in _comma_temp.itertuples():
    print(f"    comma temp files in {r.split}/{r.bearing}: {r.temp_comma} of {r.n_temp}")
_b14 = INVENTORY.set_index("bearing").loc["Bearing1_4"]
hook("delimiter_bearing1_4", _b14.acc_semicolon == _b14.n_acc > 0,
     f"Full_Test_Set/Bearing1_4: {_b14.acc_semicolon}/{_b14.n_acc} acc files detected as semicolon", stop=False)
_others = sorted(set(_semi_acc.bearing) - {"Bearing1_4"})
hook("delimiter_others", True, f"other bearings needing semicolon acc parsing: {_others or 'none'}; "
     f"temp files needing comma: {sorted(_comma_temp.bearing) or 'none'}", stop=False)

# --- acc:temp ratio (documented 10 s vs 60 s -> ~6:1) ---
_both = INVENTORY[INVENTORY.n_temp > 0]
_ratio_ok = _both["acc:temp"].between(*RATIO_TOL)
print("\nacc:temp file-count ratio (bearings with both modalities):")
for (_, r), ok in zip(_both.iterrows(), _ratio_ok):
    print(f"  {r['bearing']}: acc {r['n_acc']} / temp {r['n_temp']} = {r['acc:temp']:.2f}"
          f"{'' if ok else '   <-- outside ' + str(RATIO_TOL)}")
hook("acc_temp_ratio", _ratio_ok.all(), f"{int(_ratio_ok.sum())}/{len(_both)} bearings with both modalities have "
     f"acc:temp in {RATIO_TOL} (documented {TEMP_PERIOD_S // ACC_PERIOD_S}:1)", stop=False)

# --- zero-temperature bearings (quirk 7) ---
NO_TEMP = set(INVENTORY.loc[INVENTORY.n_temp == 0, "bearing"])
if NO_TEMP != EXPECTED_NO_TEMP:
    loud(f"ZERO-TEMPERATURE BEARING LIST DIFFERS FROM THE AUDIT.\nfound:    {sorted(NO_TEMP)}\nexpected: "
         f"{sorted(EXPECTED_NO_TEMP)}\nmissing from found: {sorted(EXPECTED_NO_TEMP - NO_TEMP)} | "
         f"unexpected: {sorted(NO_TEMP - EXPECTED_NO_TEMP)}")
hook("zero_temp_list", NO_TEMP == EXPECTED_NO_TEMP,
     f"bearings with 0 temp files: {sorted(NO_TEMP)} (expected {sorted(EXPECTED_NO_TEMP)})",
     stop=not ALLOW_NO_TEMP_MISMATCH)

# --- temp file row counts, measured (quirk 6) ---
_all_temp_len = np.concatenate([DATA[b]["temp_len"] for b in BEARING_NAMES])
TEMP_ROW_COUNTS = Counter(_all_temp_len.tolist())
if not TEMP_ROW_COUNTS:
    raise RuntimeError("No temperature files found at all.")
TEMP_LEN = int(TEMP_ROW_COUNTS.most_common(1)[0][0])
print(f"\nRows per temp file (measured, all {len(_all_temp_len):,} files): {dict(sorted(TEMP_ROW_COUNTS.items()))}")
print(f"Temperature window length = modal row count = {TEMP_LEN} samples = {TEMP_LEN / TEMP_FS:.1f} s at {TEMP_FS} Hz. "
      f"Longer files are cropped to the first {TEMP_LEN} rows; shorter files are dropped (counts in section 9).")

## Windows and degradation-stage proxy labels

A window is one record file, so it cannot cross a bearing boundary. The stage of window *i* in bearing *b* is
`min(floor(N_STAGES · (t_i − t_first(b)) / (t_last(b) − t_first(b))), N_STAGES − 1)`. It is computed **within each
bearing**, on that modality's own record timeline. For temperature, labels are computed on the bearing's full temp
timeline **before** any too-short file is dropped, so dropping a file never shifts another window's stage.

In [ ]:
# ==============================================================================
# 9. WINDOWS + PER-BEARING STAGE LABELS (+ optional vibration downsampling)
# ==============================================================================
from scipy.signal import decimate


def stage_labels(t):
    """Per-bearing equal-width elapsed-time bins. t: one bearing's record times (strictly increasing)."""
    span = t[-1] - t[0]
    assert span > 0, "a bearing needs a positive recorded span"
    frac = (t - t[0]) / span
    return np.minimum(np.floor(frac * N_STAGES).astype(np.int64), N_STAGES - 1), frac


def build_modality(modality):
    parts = defaultdict(list)
    dropped = {}
    for b in BEARING_NAMES:
        D = DATA[b]
        if modality == "vibration":
            t, nums, prefix = D["acc_t"], D["acc_num"], "acc"
            stage, frac = stage_labels(t)
            X = D["acc_x"]
            if VIB_DOWNSAMPLE_FACTOR > 1:
                X = decimate(X, VIB_DOWNSAMPLE_FACTOR, axis=1, zero_phase=True).astype(np.float32)
            keep = np.arange(len(t))
        else:
            if len(D["temp_t"]) == 0:
                continue
            t, nums, prefix = D["temp_t"], D["temp_num"], "temp"
            stage, frac = stage_labels(t)                 # full temp timeline, BEFORE dropping short files
            lens = D["temp_len"]
            offs = np.concatenate([[0], np.cumsum(lens)])
            keep = np.where(lens >= TEMP_LEN)[0]
            dropped[b] = (int((lens < TEMP_LEN).sum()), int((lens > TEMP_LEN).sum()))
            X = np.stack([D["temp_flat"][offs[i]:offs[i] + TEMP_LEN] for i in keep])[..., None]
        parts["X"].append(X)
        parts["y"].append(stage[keep])
        parts["frac"].append(frac[keep])
        parts["elapsed_s"].append((t - t[0])[keep])
        parts["file_num"].append(nums[keep])
        parts["bearing"].append(np.full(len(keep), b, dtype="<U12"))
        parts["condition"].append(np.full(len(keep), COND_OF[b], np.int64))
        parts["name"].append(np.array([f"{b}/{prefix}_{n:05d}" for n in nums[keep]]))
    M = {k: np.concatenate(v) for k, v in parts.items()}
    M["wid"] = np.arange(len(M["y"]), dtype=np.int64)
    M["bearings"] = [b for b in BEARING_NAMES if b in set(M["bearing"])]
    M["dropped"] = dropped
    return M


MOD = {m: build_modality(m) for m in ("vibration", "temperature")}
for _b in BEARING_NAMES:
    DATA[_b].pop("acc_x", None)                           # raw bursts now live in MOD["vibration"]["X"]
for _b, (_short, _long) in MOD["temperature"]["dropped"].items():
    if _short or _long:
        print(f"  temperature {_b}: {_short} files shorter than {TEMP_LEN} rows dropped, {_long} longer files cropped")

for _b, _v in CLOCK_REPAIRS.items():                    # how much the repaired clock matters for the labels
    for _kind, _key in (("acc", "acc_t"), ("temp", "temp_t")):
        if _v[_kind]:
            _st_time, _ = stage_labels(DATA[_b][_key])
            _st_order, _ = stage_labels(np.arange(len(DATA[_b][_key]), dtype=float))
            print(f"  {_b} {_kind} (clock repaired): time-based vs file-order stage labels agree on "
                  f"{(_st_time == _st_order).mean():.2%} of {len(_st_time)} files")

VIB_SEQ_LEN = MOD["vibration"]["X"].shape[1]
TEMP_SEQ_LEN = MOD["temperature"]["X"].shape[1]
N_BEARINGS = {m: len(MOD[m]["bearings"]) for m in MOD}
print(f"\nVIB_DOWNSAMPLE_FACTOR = {VIB_DOWNSAMPLE_FACTOR} -> vibration window = {VIB_SEQ_LEN} samples x "
      f"{MOD['vibration']['X'].shape[2]} channels ({ACC_ROWS / ACC_FS:.2f} s burst at {ACC_FS / VIB_DOWNSAMPLE_FACTOR:,.0f} Hz)"
      f" -> {VIB_SEQ_LEN} Mamba tokens per window")
print(f"Temperature window = {TEMP_SEQ_LEN} samples x 1 channel -> {TEMP_SEQ_LEN} Mamba tokens per window")
for m in MOD:
    print(f"{m:>11}: {len(MOD[m]['y']):,} windows from {N_BEARINGS[m]} bearings | X {MOD[m]['X'].shape} "
          f"{MOD[m]['X'].dtype} ({MOD[m]['X'].nbytes / 1e9:.2f} GB)")
hook("bearing_counts", N_BEARINGS["vibration"] == 17 and N_BEARINGS["temperature"] == 17 - len(NO_TEMP),
     f"vibration uses {N_BEARINGS['vibration']} bearings, temperature uses {N_BEARINGS['temperature']} "
     f"(vibration NOT restricted to temperature's set)")

# --- no window crosses a bearing boundary; labels are per-bearing, never pooled ---
for m, M in MOD.items():
    _blocks = [M["bearing"][0]] + [M["bearing"][i] for i in range(1, len(M["bearing"]))
                                    if M["bearing"][i] != M["bearing"][i - 1]]
    _prefix_ok = all(n.split("/")[0] == b for n, b in zip(M["name"], M["bearing"]))
    hook(f"{m}_window_bearing", _prefix_ok and len(_blocks) == len(set(_blocks)) == N_BEARINGS[m],
         f"each {m} window is one record file of one bearing; each bearing is one contiguous block")
    _ok, _table = True, []
    for b in M["bearings"]:
        sel = M["bearing"] == b
        D = DATA[b]
        t_full = D["acc_t"] if m == "vibration" else D["temp_t"]
        re_stage, _ = stage_labels(t_full)               # recomputed from THIS bearing's timestamps alone
        nums_full = D["acc_num"] if m == "vibration" else D["temp_num"]
        re_map = dict(zip(nums_full.tolist(), re_stage.tolist()))
        y_b = M["y"][sel]
        _ok &= all(re_map[n] == s for n, s in zip(M["file_num"][sel].tolist(), y_b.tolist()))
        _ok &= bool(np.all(np.diff(y_b) >= 0)) and set(y_b.tolist()) == set(range(N_STAGES))
        _ok &= re_stage[0] == 0 and re_stage[-1] == N_STAGES - 1
        _table.append([b, COND_OF[b]] + np.bincount(y_b, minlength=N_STAGES).tolist())
    hook(f"{m}_stage_labels_per_bearing", _ok,
         f"{m}: labels recomputed per bearing match, are non-decreasing in time, cover all {N_STAGES} stages, "
         f"start at Q1 and end at Q{N_STAGES} on every bearing")
    print(pd.DataFrame(_table, columns=["bearing", "cond"] + [f"Q{s + 1}" for s in range(N_STAGES)]).to_string(index=False))

In [ ]:
# ==============================================================================
# 10. LEAVE-ONE-CONDITION-OUT FOLDS + BEARING-LEVEL INNER VALIDATION
# ==============================================================================
TEMP_BEARINGS = set(MOD["temperature"]["bearings"])


def pick_val_bearings(test_c):
    """One whole bearing per TRAINING condition with >= 2 bearings. Prefer a bearing that has temperature (so both
    modalities share it) when that condition has >= 2 temperature bearings. Otherwise use a vibration-only bearing, so
    a condition's only temperature bearing stays in training. Seeded per fold; independent of the run seed."""
    rng = np.random.RandomState(INNER_VAL_SEED + test_c)
    val = []
    for tc in [c for c in FOLD_CONDITIONS if c != test_c]:
        all_b = [b for b in BEARING_NAMES if COND_OF[b] == tc]
        temp_b = [b for b in all_b if b in TEMP_BEARINGS]
        if len(temp_b) >= 2:
            pool = temp_b
        elif len(all_b) >= 2:
            pool = [b for b in all_b if b not in temp_b]
        else:
            pool = []
        if pool:
            val.append(pool[rng.randint(len(pool))])
    return sorted(val, key=BEARING_NAMES.index)


FOLDS = {}
print("Temperature bearings per condition:",
      {c: sorted(b for b in TEMP_BEARINGS if COND_OF[b] == c) for c in FOLD_CONDITIONS})
for c in FOLD_CONDITIONS:
    val_b = pick_val_bearings(c)
    for m in MOD:
        avail = MOD[m]["bearings"]
        test = [b for b in avail if COND_OF[b] == c]
        val = [b for b in avail if b in val_b]
        train = [b for b in avail if COND_OF[b] != c and b not in val_b]
        reasons = []
        if not test:
            reasons.append(f"condition {c} has no {m} bearing to test on")
        if not train:
            reasons.append(f"no {m} training bearing remains in conditions {[x for x in FOLD_CONDITIONS if x != c]}")
        if not val:
            reasons.append(f"no {m} bearing could be held out for inner validation without emptying a training condition")
        FOLDS[(m, c)] = {"train": train, "val": val, "test": test, "valid": not reasons, "reason": "; ".join(reasons)}
        assert not (set(train) & set(val) or set(train) & set(test) or set(val) & set(test))
        if not reasons:
            assert set(train) | set(val) | set(test) == set(avail)

_fold_rows = []
for (m, c), f in FOLDS.items():
    _fold_rows.append({"modality": m, "test_cond": c, "valid": f["valid"], "n_train": len(f["train"]),
                       "n_val": len(f["val"]), "n_test": len(f["test"]), "train": " ".join(f["train"]),
                       "val": " ".join(f["val"]), "test": " ".join(f["test"]), "skip_reason": f["reason"]})
with pd.option_context("display.width", 250, "display.max_colwidth", 80):
    print(pd.DataFrame(_fold_rows).to_string(index=False))
for (m, c), f in FOLDS.items():
    if not f["valid"]:
        loud(f"{m.upper()} FOLD 'test condition {c}' SKIPPED for {m} only: {f['reason']}")
hook("vibration_folds_valid", all(FOLDS[("vibration", c)]["valid"] for c in FOLD_CONDITIONS),
     "all 3 vibration folds have train / val / test bearings")
hook("temperature_folds_checked", True, f"temperature folds valid: "
     f"{[c for c in FOLD_CONDITIONS if FOLDS[('temperature', c)]['valid']]}; skipped (with reason above): "
     f"{[c for c in FOLD_CONDITIONS if not FOLDS[('temperature', c)]['valid']]}", stop=False)

In [ ]:
# ==============================================================================
# 12. TRAINING / EVALUATION UTILITIES, results CSV (resumable)
# ==============================================================================
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score

STAGES = list(range(N_STAGES))
STAGE_NAMES = [f"Q{s + 1}" for s in STAGES]


class WindowDataset(Dataset):
    def __init__(self, M, idx):
        self.X, self.y, self.idx = M["X"], M["y"], np.asarray(idx)

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        return torch.from_numpy(self.X[j]), int(self.y[j]), int(j)


def make_loader(M, idx, shuffle, seed):
    return DataLoader(WindowDataset(M, idx), batch_size=BATCH_SIZE if shuffle else EVAL_BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True, worker_init_fn=worker_init_fn,
                      generator=make_generator(seed) if shuffle else None)


NORM_STATS = {}


def norm_stats(modality, test_c, train_idx):
    """Per-channel mean / SD over the fold's TRAINING bearings only."""
    key = (modality, test_c)
    if key not in NORM_STATS:
        X = MOD[modality]["X"]
        s = np.zeros(X.shape[2])
        ss = np.zeros(X.shape[2])
        n = 0
        for chunk in np.array_split(train_idx, max(1, len(train_idx) // 1024)):
            x = X[chunk].astype(np.float64)
            s += x.sum(axis=(0, 1))
            ss += (x ** 2).sum(axis=(0, 1))
            n += x.shape[0] * x.shape[1]
        mu = s / n
        sd = np.sqrt(np.maximum(ss / n - mu ** 2, 0.0)) + NORM_EPS
        NORM_STATS[key] = (mu.astype(np.float32), sd.astype(np.float32))
    return NORM_STATS[key]


def compute_metrics(y, P):
    pred = P.argmax(1)
    cm = confusion_matrix(y, pred, labels=STAGES)
    rec = np.where(cm.sum(1) > 0, np.diag(cm) / np.maximum(cm.sum(1), 1), np.nan)
    out = {"accuracy": float(accuracy_score(y, pred)),
           "macro_f1": float(f1_score(y, pred, average="macro", labels=STAGES, zero_division=0)),
           "balanced_accuracy": float(balanced_accuracy_score(y, pred))}
    out.update({f"recall_{STAGE_NAMES[s]}": float(rec[s]) for s in STAGES})
    out["confusion_matrix"] = cm
    return out


@torch.no_grad()
def predict(model, loader, mu, sd):
    model.eval()
    probs, ys, wids = [], [], []
    for x, y, j in loader:
        x = (x.to(device, non_blocking=True) - mu) / sd
        probs.append(F.softmax(model(x), dim=-1).float().cpu().numpy())
        ys.append(y.numpy())
        wids.append(j.numpy())
    return np.concatenate(probs), np.concatenate(ys), np.concatenate(wids)


def run_tag(modality, arch, test_c, seed):
    return f"{modality}_{arch}_testC{test_c}_{seed}"


def probs_path(modality, arch, test_c, seed):
    return PROBS_DIR / f"{run_tag(modality, arch, test_c, seed)}.npz"


def ckpt_path(modality, arch, test_c, seed):
    return CKPT_DIR / f"{run_tag(modality, arch, test_c, seed)}.pt"


def done_keys():
    """Runs whose CSV row AND prediction file both exist under the current CONFIG_TAG."""
    if not RESULTS_CSV.exists():
        return set()
    df = pd.read_csv(RESULTS_CSV)
    keys = set()
    for r in df[df.config_tag == CONFIG_TAG].itertuples():
        p = probs_path(r.modality, r.arch, int(r.test_condition), int(r.seed))
        if p.exists():
            with np.load(p, allow_pickle=False) as z:
                if str(z["config_tag"]) == CONFIG_TAG:
                    keys.add((r.modality, r.arch, int(r.test_condition), int(r.seed)))
    return keys


def append_result(row):
    new = pd.DataFrame([row])
    if RESULTS_CSV.exists():
        old = pd.read_csv(RESULTS_CSV)
        stale = ((old.config_tag == row["config_tag"]) & (old.modality == row["modality"]) & (old.arch == row["arch"])
                 & (old.test_condition == row["test_condition"]) & (old.seed == row["seed"]))
        new = pd.concat([old[~stale], new], ignore_index=True)
    tmp = RESULTS_CSV.with_suffix(".tmp")
    new.to_csv(tmp, index=False)
    tmp.replace(RESULTS_CSV)


def run_one(modality, arch, test_c, seed):
    t0 = time.time()
    M, fold = MOD[modality], FOLDS[(modality, test_c)]
    idx = {s: np.where(np.isin(M["bearing"], fold[s]))[0] for s in ("train", "val", "test")}
    mu_np, sd_np = norm_stats(modality, test_c, idx["train"])
    mu = torch.tensor(mu_np, device=device).view(1, 1, -1)
    sd = torch.tensor(sd_np, device=device).view(1, 1, -1)

    seed_all(seed)                                           # at the START of every run, before the model is built
    model = build_model(modality, arch).to(device)
    if arch == "mamba":
        assert all(type(b.mamba) is MAMBA_CLASS for b in model.mamba_blocks)
    counts = np.bincount(M["y"][idx["train"]], minlength=N_STAGES)
    assert (counts > 0).all(), f"a stage is missing from training: {counts}"
    cw = torch.tensor(len(idx["train"]) / (N_STAGES * counts), dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=cw)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE[arch], weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
    tr = make_loader(M, idx["train"], True, seed)
    va = make_loader(M, idx["val"], False, seed)
    te = make_loader(M, idx["test"], False, seed)
    tag = f"[{modality}/{arch} test=C{test_c} seed {seed}]"
    print(f"{tag} {N_PARAMS[(modality, arch)]:,} params | bearings train {len(fold['train'])} / val "
          f"{len(fold['val'])} {fold['val']} / test {len(fold['test'])} | windows train {len(idx['train'])} / val "
          f"{len(idx['val'])} / test {len(idx['test'])} | seq len {SEQ_LEN[modality]}")

    best_f1, best_state, best_epoch = -1.0, None, None
    for epoch in range(1, EPOCHS + 1):
        te0 = time.time()
        model.train()
        tot, nb = 0.0, 0
        for x, y, _ in tr:
            x = (x.to(device, non_blocking=True) - mu) / sd
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(x), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            tot += loss.item()
            nb += 1
        scheduler.step()
        if not math.isfinite(tot):
            raise RuntimeError(f"{tag} non-finite training loss at epoch {epoch}")
        vP, vy, _ = predict(model, va, mu, sd)
        vf1 = compute_metrics(vy, vP)["macro_f1"]
        if vf1 > best_f1:
            best_f1, best_state, best_epoch = vf1, copy.deepcopy(model.state_dict()), epoch
        print(f"  epoch {epoch:2d} | loss {tot / nb:.4f} | val macro-F1 {vf1:.4f}{' *' if epoch == best_epoch else ''} "
              f"| {time.time() - te0:.0f} s")
        if epoch == 1:
            print(f"  (estimated run time ≈ {(time.time() - te0) * EPOCHS / 60:.1f} min)")
        if epoch - best_epoch >= EARLY_STOP_PATIENCE:
            print(f"  early stop at epoch {epoch}, no improvement for {EARLY_STOP_PATIENCE} epochs")
            break

    model.load_state_dict(best_state)
    P, y, wid = predict(model, te, mu, sd)
    assert np.array_equal(wid, idx["test"]) and np.array_equal(y, M["y"][wid])
    met = compute_metrics(y, P)
    np.savez(probs_path(modality, arch, test_c, seed),
             window_id=wid, window_name=M["name"][wid], bearing=M["bearing"][wid], condition=M["condition"][wid],
             stage_label=y, probs=P.astype(np.float32), pred=P.argmax(1), frac_of_life=M["frac"][wid],
             file_num=M["file_num"][wid], config_tag=np.array(CONFIG_TAG), modality=np.array(modality),
             arch=np.array(arch), test_condition=np.int64(test_c), seed=np.int64(seed))
    torch.save({"state_dict": best_state, "best_epoch": best_epoch, "best_val_macro_f1": best_f1,
                "norm_mu": mu_np, "norm_sd": sd_np, "fold": fold, "config_tag": CONFIG_TAG,
                "modality": modality, "arch": arch, "test_condition": test_c, "seed": seed},
               ckpt_path(modality, arch, test_c, seed))
    row = {"config_tag": CONFIG_TAG, "modality": modality, "arch": arch, "test_condition": test_c, "seed": seed,
           "n_bearings_modality": N_BEARINGS[modality], "n_train_bearings": len(fold["train"]),
           "n_val_bearings": len(fold["val"]), "n_test_bearings": len(fold["test"]),
           "train_bearings": " ".join(fold["train"]), "val_bearings": " ".join(fold["val"]),
           "test_bearings": " ".join(fold["test"]), "n_train_windows": len(idx["train"]),
           "n_val_windows": len(idx["val"]), "n_test_windows": len(idx["test"]), "seq_len": SEQ_LEN[modality],
           "n_params": N_PARAMS[(modality, arch)], "best_epoch": best_epoch, "best_val_macro_f1": best_f1,
           **{k: v for k, v in met.items() if k != "confusion_matrix"},
           "confusion_matrix": json.dumps(met["confusion_matrix"].tolist()),
           "runtime_s": round(time.time() - t0, 1), "finished_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    append_result(row)
    print(f"{tag} TEST: acc {met['accuracy']:.4f} | macro-F1 {met['macro_f1']:.4f} | bal-acc "
          f"{met['balanced_accuracy']:.4f} | recall " + " ".join(f"{n} {met['recall_' + n]:.3f}" for n in STAGE_NAMES)
          + f" | best epoch {best_epoch} | {time.time() - t0:.0f} s")
    del model, best_state
    torch.cuda.empty_cache()

In [ ]:
# ==============================================================================
# F2. PROVENANCE — ported cells vs the baseline notebook (if its file is reachable; build-time verified otherwise)
# ==============================================================================
PORTED_CELL_SHA1 = {
    "# 1. SETUP + CONSTANTS — every fixed value lives here": "0677d0d6ea2e",
    "# 2. REAL mamba-ssm — install if missing, import, raise on failure (no fallback)": "ee529627d6c7",
    "# 3. SEEDING, DETERMINISM FLAGS, HOOK HELPER": "25054e2867c7",
    "## Data acquisition: download → recursive extraction → split discovery": "0fe9801343b0",
    "# 4. DOWNLOAD (PHM Society mirror of the NASA Prognostics Data Repository)": "af34f325387d",
    "# 5. RECURSIVE EXTRACTION — extract every .zip found, in passes, until none remain": "65e2d7eb232c",
    "# 6. SPLIT DISCOVERY — Learning_set + Full_Test_Set only; Test_set located and EXCLUDED": "ccbd0f1ecece",
    "## Parsing: per-file delimiter auto-detection, shape/NaN checks, timestamps": "4b7643e06c71",
    "# 7. RAW CSV PARSING (guarded against Test_set, delimiter auto-detected per file, cached)": "c56c60eb54c3",
    "# 8. INVENTORY + VERIFICATION: delimiters, shape/NaN, monotonic time, acc:temp ratio, zero-temperature bearings": "7f8813beb204",
    "## Windows and degradation-stage proxy labels": "468372ad0bab",
    "# 9. WINDOWS + PER-BEARING STAGE LABELS (+ optional vibration downsampling)": "fdad19fb4b36",
    "# 10. LEAVE-ONE-CONDITION-OUT FOLDS + BEARING-LEVEL INNER VALIDATION": "c387f26f37d7",
    "# 12. TRAINING / EVALUATION UTILITIES, results CSV (resumable)": "5a9dbd3c3d8a"
}
_cands = [Path("femto-baselines.ipynb"), Path("FEMTO/femto-baselines.ipynb"), Path("../FEMTO/femto-baselines.ipynb")]
if Path("/kaggle/input").is_dir():
    _cands += sorted(Path("/kaggle/input").glob("**/femto-baselines.ipynb"))
_found = next((p for p in _cands if p.is_file()), None)
if _found is None:
    print("Baseline notebook file not reachable from this session; the ported cells were verified byte-identical "
          "when this notebook was generated (SHA-1 list above).")
else:
    _bsrc = ["".join(c["source"]) for c in json.load(open(_found))["cells"]]
    _mismatch = []
    for _h, _sha in PORTED_CELL_SHA1.items():
        _hit = [s for s in _bsrc if _h in s.split("\n")[:3]]
        if len(_hit) != 1 or hashlib.sha1(_hit[0].encode()).hexdigest()[:12] != _sha:
            _mismatch.append(_h)
    hook("ported_cells_match_baseline", not _mismatch, f"{len(PORTED_CELL_SHA1) - len(_mismatch)}/{len(PORTED_CELL_SHA1)} "
         f"ported cells identical to {_found}; differing: {_mismatch or 'none'}", stop=False)

## STEP 1 — per-burst token features

One vibration burst → 8 features. One temperature record → 2 features. The baseline's `MOD["vibration"]["X"]` still
holds every raw burst in file order (`VIB_DOWNSAMPLE_FACTOR = 1`), and `DATA[b]["temp_flat"]` holds every temperature
record, short ones included.

In [ ]:
# ==============================================================================
# F3. STEP 1 — PER-BURST TOKEN FEATURES
# ==============================================================================
FUSION_BEARINGS = list(MOD["temperature"]["bearings"])            # bearings with BOTH modalities (baseline's list)
N_FUSION_BEARINGS = len(FUSION_BEARINGS)
hook("fusion_bearings", N_FUSION_BEARINGS == 17 - len(EXPECTED_NO_TEMP) and not set(FUSION_BEARINGS) & EXPECTED_NO_TEMP,
     f"{N_FUSION_BEARINGS} bearings have both modalities = 17 minus the baseline's EXPECTED_NO_TEMP "
     f"{sorted(EXPECTED_NO_TEMP)}: {FUSION_BEARINGS}")

VIB_FEATURE_NAMES = [f"{f}_{c}" for c in ("horiz", "vert") for f in ("rms", "kurtosis", "peak", "crest")]
TEMP_FEATURE_NAMES = ["temp_mean_C", "temp_slope_C_per_s"]


def vib_burst_features(X, chunk=512):
    """X (n, 2560, 2) raw g -> (n, 8): per channel [RMS, kurtosis (Pearson), peak |x|, crest = peak / RMS]."""
    out = np.empty((len(X), X.shape[2], 4), np.float64)
    for lo in range(0, len(X), chunk):
        x = X[lo:lo + chunk].astype(np.float64)
        rms = np.sqrt((x ** 2).mean(axis=1))
        peak = np.abs(x).max(axis=1)
        xc = x - x.mean(axis=1, keepdims=True)
        m2 = (xc ** 2).mean(axis=1)
        kurt = (xc ** 4).mean(axis=1) / np.maximum(m2 ** 2, FEAT_EPS)
        crest = peak / np.maximum(rms, FEAT_EPS)
        out[lo:lo + chunk] = np.stack([rms, kurt, peak, crest], axis=-1)
    return out.reshape(len(X), -1)                               # order = VIB_FEATURE_NAMES


def temp_burst_features(v):
    """v: one record's temperatures at TEMP_FS Hz -> (mean degC, least-squares slope degC/s)."""
    t = np.arange(len(v)) / TEMP_FS
    tc, vc = t - t.mean(), v - v.mean()
    return v.mean(), (tc * vc).sum() / (tc ** 2).sum()


TOK = {}
_mv = MOD["vibration"]
_t = time.time()
for b in FUSION_BEARINGS:
    sel = np.where(_mv["bearing"] == b)[0]
    assert np.array_equal(_mv["file_num"][sel], DATA[b]["acc_num"]), f"{b}: vibration rows are not in file order"
    D = DATA[b]
    offs = np.concatenate([[0], np.cumsum(D["temp_len"])])
    keep = np.where(D["temp_len"] >= MIN_TEMP_ROWS)[0]
    tf = np.array([temp_burst_features(D["temp_flat"][offs[i]:offs[i + 1]].astype(np.float64)) for i in keep])
    TOK[b] = {"vib_raw": vib_burst_features(_mv["X"][sel]), "vib_t": D["acc_t"].copy(), "vib_num": D["acc_num"].copy(),
              "temp_raw": tf.reshape(-1, N_TEMP_FEAT), "temp_t": D["temp_t"][keep].copy(),
              "temp_num": D["temp_num"][keep].copy(), "temp_len": D["temp_len"][keep].copy(),
              "temp_skipped": int(len(D["temp_len"]) - len(keep))}
print(f"Features for {N_FUSION_BEARINGS} bearings in {time.time() - _t:.0f} s")

_all_v = np.concatenate([TOK[b]["vib_raw"] for b in FUSION_BEARINGS])
_all_t = np.concatenate([TOK[b]["temp_raw"] for b in FUSION_BEARINGS])
hook("token_features_finite", np.isfinite(_all_v).all() and np.isfinite(_all_t).all() and (_all_v > 0).all(),
     f"{len(_all_v):,} vibration tokens x {N_VIB_FEAT} and {len(_all_t):,} temperature tokens x {N_TEMP_FEAT} features, "
     f"all finite; vibration features all > 0 (log-safe)")
print(pd.DataFrame(np.percentile(_all_v, [0, 50, 99, 100], axis=0), index=["min", "median", "p99", "max"],
                   columns=VIB_FEATURE_NAMES).round(4).to_string())
print(pd.DataFrame(np.percentile(_all_t, [0, 50, 99, 100], axis=0), index=["min", "median", "p99", "max"],
                   columns=TEMP_FEATURE_NAMES).round(5).to_string())
print("temperature records skipped as tokens (< MIN_TEMP_ROWS rows):",
      {b: TOK[b]["temp_skipped"] for b in FUSION_BEARINGS if TOK[b]["temp_skipped"]} or "none")

## STEP 2 — timestamp anchor correction

t = 0 is each bearing's **first vibration burst**, for both modalities. The table shows, per bearing:
- the measured start offset, with the 1–107 s range check;
- the end gap (temperature end − vibration end; both streams stop when the test stops);
- which clock model the end gap supports;
- the correction actually applied to the temperature timestamps under `ANCHOR_MODE`.

In [ ]:
# ==============================================================================
# F4. STEP 2 — TIMESTAMP ANCHOR CORRECTION (verified and printed, never assumed)
# ==============================================================================
_rows = []
for b in FUSION_BEARINGS:
    D = DATA[b]
    vib0 = D["acc_t"][0]
    start_off = D["temp_t"][0] - vib0                                    # all temp files, incl. any skipped as tokens
    vib_end = D["acc_t"][-1] + ACC_ROWS / ACC_FS
    temp_end = D["temp_t"][-1] + D["temp_len"][-1] / TEMP_FS
    end_gap = temp_end - vib_end
    if abs(end_gap) <= CLOCK_EVIDENCE_TOL_S:
        evidence = "shared_clock"
    elif abs(end_gap - start_off) <= CLOCK_EVIDENCE_TOL_S:
        evidence = "subtract_offset"
    else:
        evidence = "ambiguous"
    correction = 0.0 if ANCHOR_MODE == "shared_clock" else -start_off
    TOK[b]["vib_tau"] = TOK[b]["vib_t"] - vib0
    TOK[b]["temp_tau"] = TOK[b]["temp_t"] - vib0 + correction
    lo, hi = OFFSET_EXPECTED_RANGE_S
    dist = max(lo - start_off, start_off - hi, 0.0)
    _rows.append({"bearing": b, "cond": COND_OF[b], "start_offset_s": round(start_off, 2),
                  "in_1_107s": dist == 0, "outside_by_s": round(dist, 2), "end_gap_s": round(end_gap, 2),
                  "evidence": evidence, "temp_correction_applied_s": round(correction, 2),
                  "first_temp_tau_s": round(TOK[b]["temp_tau"][0], 2), "vib_life_min": round(TOK[b]["vib_tau"][-1] / 60, 1)})
ANCHOR_TABLE = pd.DataFrame(_rows)
print(f"ANCHOR_MODE = {ANCHOR_MODE!r}: t = 0 is each bearing's first vibration burst; temperature tau = temp_raw - "
      f"vib_first{' - start_offset' if ANCHOR_MODE == 'subtract_offset' else ''}")
print(ANCHOR_TABLE.to_string(index=False))

_near = ANCHOR_TABLE[(ANCHOR_TABLE.outside_by_s > 0) & (ANCHOR_TABLE.outside_by_s <= OFFSET_FAR_MARGIN_S)]
_far = ANCHOR_TABLE[ANCHOR_TABLE.outside_by_s > OFFSET_FAR_MARGIN_S]
if len(_far):
    loud("START OFFSET FAR OUTSIDE THE AUDITED 1-107 s RANGE — investigate these bearings before trusting their tokens:\n"
         + _far[["bearing", "start_offset_s", "outside_by_s"]].to_string(index=False))
hook("anchor_offsets_near_range", not len(_far),
     f"no bearing's start offset is more than {OFFSET_FAR_MARGIN_S:.0f} s outside {OFFSET_EXPECTED_RANGE_S}; far: "
     f"{_far.bearing.tolist() or 'none'}", stop=not ALLOW_FAR_OFFSETS)
hook("anchor_offsets_in_range", not len(_near), f"slightly outside {OFFSET_EXPECTED_RANGE_S} (<= "
     f"{OFFSET_FAR_MARGIN_S:.0f} s): {dict(zip(_near.bearing, _near.start_offset_s)) or 'none'}", stop=False)

_ev = Counter(ANCHOR_TABLE.evidence)
print(f"\nClock-model evidence from end gaps: {dict(_ev)}")
_against = _ev.get("subtract_offset" if ANCHOR_MODE == "shared_clock" else "shared_clock", 0)
_for = _ev.get(ANCHOR_MODE, 0)
if _against > _for:
    loud(f"END-GAP EVIDENCE FAVOURS {'subtract_offset' if ANCHOR_MODE == 'shared_clock' else 'shared_clock'!r} "
         f"({_against} bearings) OVER THE CHOSEN ANCHOR_MODE {ANCHOR_MODE!r} ({_for} bearings).\n"
         f"Consider switching ANCHOR_MODE in the fusion constants cell (it is part of the config tag).")
hook("anchor_mode_vs_evidence", _against <= _for, f"ANCHOR_MODE {ANCHOR_MODE!r} supported by {_for} bearings, "
     f"contradicted by {_against}, ambiguous {_ev.get('ambiguous', 0)}", stop=False)

## STEP 3 — trajectory windows, labels and token density

In [ ]:
# ==============================================================================
# F5. STEP 3 — WINDOWS (variable token counts) + CENTRE LABELS via the baseline's stage_labels
# ==============================================================================
def _vib_transform(f):
    return np.log(f) if VIB_FEATURE_LOG else f


# global token tables: bearing-major, time-sorted -> each window is one contiguous slice per modality
VT = {"feat": np.concatenate([_vib_transform(TOK[b]["vib_raw"]) for b in FUSION_BEARINGS]).astype(np.float32),
      "tau": np.concatenate([TOK[b]["vib_tau"] for b in FUSION_BEARINGS]),
      "bearing": np.concatenate([np.full(len(TOK[b]["vib_tau"]), b, "<U12") for b in FUSION_BEARINGS]),
      "num": np.concatenate([TOK[b]["vib_num"] for b in FUSION_BEARINGS])}
TT = {"feat": np.concatenate([TOK[b]["temp_raw"] for b in FUSION_BEARINGS]).astype(np.float32),
      "tau": np.concatenate([TOK[b]["temp_tau"] for b in FUSION_BEARINGS]),
      "bearing": np.concatenate([np.full(len(TOK[b]["temp_tau"]), b, "<U12") for b in FUSION_BEARINGS]),
      "num": np.concatenate([TOK[b]["temp_num"] for b in FUSION_BEARINGS])}
_v_off = dict(zip(FUSION_BEARINGS, np.concatenate([[0], np.cumsum([len(TOK[b]["vib_tau"]) for b in FUSION_BEARINGS])])))
_t_off = dict(zip(FUSION_BEARINGS, np.concatenate([[0], np.cumsum([len(TOK[b]["temp_tau"]) for b in FUSION_BEARINGS])])))

W_S, STRIDE_S = WINDOW_MINUTES * 60.0, WINDOW_STRIDE_MINUTES * 60.0
_w = defaultdict(list)
_dens, _dropped = [], {}
for b in FUSION_BEARINGS:
    vt, tt = TOK[b]["vib_tau"], TOK[b]["temp_tau"]
    L = vt[-1]                                                   # vibration-defined life (s), t = 0 at first burst
    if L <= W_S:
        starts = [0.0]
    else:
        starts = list(np.arange(0.0, L - W_S + 1e-6, STRIDE_S))
        if L - W_S - starts[-1] > 1e-6:
            starts.append(L - W_S)                               # right-aligned last window: end of life covered
    rows, n_drop = [], 0
    for k, s in enumerate(starts):
        e = min(s + W_S, L)
        side = "right" if k == len(starts) - 1 else "left"       # the last window includes the final burst at t = L
        vlo, vhi = np.searchsorted(vt, s, "left"), np.searchsorted(vt, e, side)
        tlo, thi = np.searchsorted(tt, s, "left"), np.searchsorted(tt, e, side)
        if vhi - vlo < MIN_TOKENS_PER_MODALITY or thi - tlo < MIN_TOKENS_PER_MODALITY:
            n_drop += 1
            continue
        rows.append((s, e, (s + e) / 2, vlo, vhi, tlo, thi))
    _dropped[b] = n_drop
    centres = np.array([r[2] for r in rows])
    stage, frac = stage_labels(np.concatenate([[0.0], centres, [L]]))   # the BASELINE's function, unchanged
    stage, frac = stage[1:-1], frac[1:-1]
    for (s, e, c, vlo, vhi, tlo, thi), st, fr in zip(rows, stage, frac):
        _w["bearing"].append(b); _w["condition"].append(COND_OF[b]); _w["start_s"].append(s); _w["end_s"].append(e)
        _w["centre_s"].append(c); _w["label"].append(st); _w["frac"].append(fr)
        _w["vlo"].append(_v_off[b] + vlo); _w["vhi"].append(_v_off[b] + vhi)
        _w["tlo"].append(_t_off[b] + tlo); _w["thi"].append(_t_off[b] + thi)
    nv = np.array([r[4] - r[3] for r in rows]); nt = np.array([r[6] - r[5] for r in rows])
    _dens.append({"bearing": b, "cond": COND_OF[b], "life_min": round(L / 60, 1), "windows": len(rows),
                  "dropped": n_drop, "vib_tok_med": float(np.median(nv)) if len(nv) else np.nan,
                  "vib_tok_min": int(nv.min()) if len(nv) else 0, "vib_tok_max": int(nv.max()) if len(nv) else 0,
                  "temp_tok_med": float(np.median(nt)) if len(nt) else np.nan,
                  "temp_tok_min": int(nt.min()) if len(nt) else 0, "temp_tok_max": int(nt.max()) if len(nt) else 0,
                  **{STAGE_NAMES[q]: int((stage == q).sum()) for q in STAGES}})
WIN = {k: np.array(v) for k, v in _w.items()}
WIN["wid"] = np.arange(len(WIN["label"]))
WIN["n_vib"] = WIN["vhi"] - WIN["vlo"]
WIN["n_temp"] = WIN["thi"] - WIN["tlo"]
DENSITY_TABLE = pd.DataFrame(_dens)
print(f"Windows: {WINDOW_MINUTES} min, stride {WINDOW_STRIDE_MINUTES} min -> {len(WIN['label']):,} windows over "
      f"{N_FUSION_BEARINGS} bearings. Expected density at the documented 10 s / 60 s periods: ~{W_S / ACC_PERIOD_S:.0f} "
      f"vibration and ~{W_S / TEMP_PERIOD_S:.0f} temperature tokens per full window. MEASURED:")
print(DENSITY_TABLE.to_string(index=False))
print(f"All windows: vibration tokens median {np.median(WIN['n_vib']):.0f} (min {WIN['n_vib'].min()}, max "
      f"{WIN['n_vib'].max()}); temperature tokens median {np.median(WIN['n_temp']):.0f} (min {WIN['n_temp'].min()}, "
      f"max {WIN['n_temp'].max()})")

hook("windows_every_bearing", (DENSITY_TABLE.windows > 0).all(),
     f"every one of the {N_FUSION_BEARINGS} bearings has > 0 windows; fewest: "
     f"{DENSITY_TABLE.sort_values('windows').iloc[0].bearing} ({int(DENSITY_TABLE.windows.min())} windows, "
     f"{DENSITY_TABLE.sort_values('windows').iloc[0].life_min} min life)")
_cross = 0
for w in range(len(WIN["label"])):
    b, s, e = WIN["bearing"][w], WIN["start_s"][w], WIN["end_s"][w]
    for T_, lo, hi in ((VT, WIN["vlo"][w], WIN["vhi"][w]), (TT, WIN["tlo"][w], WIN["thi"][w])):
        if not ((T_["bearing"][lo:hi] == b).all() and (T_["tau"][lo:hi] >= s).all() and (T_["tau"][lo:hi] <= e).all()):
            _cross += 1
hook("no_window_crosses_bearing", _cross == 0, f"every window's vibration AND temperature tokens belong to its own "
     f"bearing and lie inside its [start, end]; violations: {_cross}")
_lab_ok, _agree = True, []
for b in FUSION_BEARINGS:
    sel = WIN["bearing"] == b
    L = TOK[b]["vib_tau"][-1]
    ref, _ = stage_labels(np.concatenate([[0.0], WIN["centre_s"][sel], [L]]))
    _lab_ok &= np.array_equal(ref[1:-1], WIN["label"][sel]) and bool(np.all(np.diff(WIN["label"][sel]) >= 0))
    vib_lab, _ = stage_labels(TOK[b]["vib_tau"])                 # baseline's per-burst vibration labels
    nearest = np.clip(np.searchsorted(TOK[b]["vib_tau"], WIN["centre_s"][sel]), 0, len(vib_lab) - 1)
    _agree.append((vib_lab[nearest] == WIN["label"][sel]).mean())
hook("window_labels", _lab_ok, f"window labels = baseline stage_labels at the window centre, per bearing, "
     f"non-decreasing in time")
hook("window_labels_vs_baseline_bursts", min(_agree) >= 0.95, f"agreement with the baseline label of the burst at "
     f"each window centre: min {min(_agree):.3f}, mean {np.mean(_agree):.3f} (differences only at stage boundaries)",
     stop=False)
_missing = DENSITY_TABLE[(DENSITY_TABLE[STAGE_NAMES] == 0).any(axis=1)].bearing.tolist()
hook("all_stages_per_bearing", not _missing, f"bearings missing a stage among their windows: {_missing or 'none'}",
     stop=False)

## Architecture (STEPS 4–6)

```
vib tokens (8)  ─ Linear → Z̄_v ┐                         ┌→ H̃_v ─┐
                               ├ HTT: LN(Z̄_m + e_m [+ φ(τ)]) ┤  LACA   ├─ [H̃_v ; SEP ; H̃_t] (packed, padded at end) → 2× Mamba → masked attention pool → Linear(N_STAGES)
temp tokens (2) ─ Linear → Z̄_t ┘                         └→ H̃_t ─┘
```

LACA, for each direction m←n with 4 heads and masked keys:
- `A_mn = softmax(Q_m K_nᵀ/√d_h + β · LN_grid,masked(g_mn(τ_m^i − τ_n^j)))`
- `H̃_m = H_m + γ_mn · out_proj(A_mn V_n)`

τ is minutes since the window start. Packing places each sample's valid vibration tokens, then the SEP token, then
its valid temperature tokens, with all padding **after** them. Mamba is causal, so trailing padding cannot influence
any valid position, and pooling masks the padding out. A hook checks padding invariance numerically.

In [ ]:
# ==============================================================================
# F6. MODELS — HTT / masked LACA / Mamba fusion; parameter accounting; real-Mamba + padding-invariance checks
# ==============================================================================
class ContinuousTimeEmbedding(nn.Module):
    """Prior TII phi(tau): sinusoids of continuous tau at fixed frequencies -> 2-layer MLP (verbatim)."""

    def __init__(self, d_model: int, num_frequencies: int = TIME_EMBED_FREQUENCIES):
        super().__init__()
        self.frequencies = nn.Parameter(
            torch.exp(torch.linspace(0, math.log(1000.0), num_frequencies)), requires_grad=False
        )
        self.mlp = nn.Sequential(
            nn.Linear(2 * num_frequencies, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tau: torch.Tensor) -> torch.Tensor:
        if tau.dim() == 2:
            tau = tau.unsqueeze(-1)
        angles = tau * self.frequencies * 2.0 * math.pi
        return self.mlp(torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1))


class HeterogeneousTemporalTokenization(nn.Module):
    """token = LayerNorm(features + modality embedding [+ time embedding(tau)]); modality 0 = vibration, 1 = temperature.
    time_enabled False skips the time embedding entirely; the module is still built so every condition draws the same RNG."""

    def __init__(self, d_model: int, time_enabled: bool = True):
        super().__init__()
        self.d_model = d_model
        self.time_enabled = time_enabled
        self.modality_embeddings = nn.Embedding(2, d_model)
        self.time_embedding = ContinuousTimeEmbedding(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z: torch.Tensor, tau: torch.Tensor, modality_idx: int) -> torch.Tensor:
        mod_idx = torch.tensor(modality_idx, device=z.device, dtype=torch.long)
        e_m = self.modality_embeddings(mod_idx).view(1, 1, self.d_model)
        if not self.time_enabled:
            return self.norm(z + e_m)
        return self.norm(z + e_m + self.time_embedding(tau))


class MaskedGridLayerNorm(nn.Module):
    """Prior GridLayerNorm (normalize each window's whole (T_m, T_n) bias grid, no affine parameters), restricted to
    the VALID (query, key) pairs of a padded window. Padded entries are returned as 0. With no padding it equals
    F.layer_norm over the last two dimensions."""

    def __init__(self, eps: float = LAG_BIAS_LN_EPS):
        super().__init__()
        self.eps = eps

    def forward(self, x, valid):                                  # x, valid: (B, T_m, T_n)
        v = valid.to(x.dtype)
        n = v.sum(dim=(-2, -1), keepdim=True).clamp_min(1.0)
        mu = (x * v).sum(dim=(-2, -1), keepdim=True) / n
        var = (((x - mu) * v) ** 2).sum(dim=(-2, -1), keepdim=True) / n
        return (x - mu) / torch.sqrt(var + self.eps) * v


class PairwiseLACA(nn.Module):
    """Cross-attention m <- n with a learned lag bias g(delta tau) added to the scores (scaled by beta); key padding masked."""

    def __init__(self, d_model: int, num_heads: int = LACA_NUM_HEADS, beta: float = LACA_BETA):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.beta = beta
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.lag_bias = nn.Sequential(nn.Linear(1, LAG_BIAS_HIDDEN), nn.Tanh(), nn.Linear(LAG_BIAS_HIDDEN, 1))
        nn.init.zeros_(self.lag_bias[-1].weight)                  # v18 zero-init: g = 0 at start; draws no RNG
        nn.init.zeros_(self.lag_bias[-1].bias)
        self.lag_bias_norm = MaskedGridLayerNorm()

    def lag_bias_term(self, tau_m, tau_n, valid):
        delta_tau = (tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)    # (B, T_m, T_n, 1), minutes
        g = self.lag_bias(delta_tau).squeeze(-1)
        return self.lag_bias_norm(g, valid).unsqueeze(1)                       # (B, 1, T_m, T_n)

    def forward(self, H_m, H_n, tau_m, tau_n, mask_m, mask_n):
        B, T_m, _ = H_m.shape
        T_n = H_n.shape[1]
        Q = self.w_q(H_m).view(B, T_m, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)
        valid = mask_m.unsqueeze(2) & mask_n.unsqueeze(1)
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale + self.beta * self.lag_bias_term(tau_m, tau_n, valid)
        scores = scores.masked_fill(~mask_n[:, None, None, :], float("-inf"))  # every window has >= 1 key per modality
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T_m, self.d_model)
        out = self.out_proj(out) * mask_m.unsqueeze(-1).to(out.dtype)
        tau_diff = tau_n.unsqueeze(1) - tau_m.unsqueeze(2)                      # key time - query time (minutes)
        per_q = (attn.mean(dim=1) * tau_diff).sum(dim=-1)
        expected_lag = (per_q * mask_m).sum(dim=1) / mask_m.sum(dim=1).clamp_min(1)
        return out, expected_lag


class BiModalLACAFusion(nn.Module):
    def __init__(self, d_model: int, beta: float = LACA_BETA):
        super().__init__()
        self.laca_vt = PairwiseLACA(d_model, beta=beta)          # vibration queries attend to temperature
        self.laca_tv = PairwiseLACA(d_model, beta=beta)          # temperature queries attend to vibration
        self.gamma_vt = nn.Parameter(torch.tensor(GAMMA_INIT))
        self.gamma_tv = nn.Parameter(torch.tensor(GAMMA_INIT))

    def forward(self, Hv, Ht, tv, tt, mv, mt):
        Hv_from_t, lag_vt = self.laca_vt(Hv, Ht, tv, tt, mv, mt)
        Ht_from_v, lag_tv = self.laca_tv(Ht, Hv, tt, tv, mt, mv)
        return Hv + self.gamma_vt * Hv_from_t, Ht + self.gamma_tv * Ht_from_v, {"lag_vt": lag_vt, "lag_tv": lag_tv}


class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba selective-scan SSM."""

    def __init__(self, d_model, d_state, d_conv, expand):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = MAMBA_CLASS(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x):
        return x + self.mamba(self.norm(x))


class FEMTOFusionMamba(nn.Module):
    """Per-burst features -> Linear -> HTT -> [LACA] -> packed [vib ; SEP ; temp] -> 2 Mamba blocks -> masked attention
    pooling -> Linear(N_STAGES). Construction order is identical for A/B/C/D (same initial weights per seed)."""

    def __init__(self, htt_time, laca_lag, cross_attention=True):
        super().__init__()
        self.htt_time_enabled, self.laca_lag_enabled, self.cross_attention = htt_time, laca_lag, cross_attention
        self.vib_proj = nn.Linear(N_VIB_FEAT, D_MODEL)
        self.temp_proj = nn.Linear(N_TEMP_FEAT, D_MODEL)
        self.htt = HeterogeneousTemporalTokenization(D_MODEL, time_enabled=htt_time)
        self.laca = BiModalLACAFusion(D_MODEL, beta=LACA_BETA if laca_lag else 0.0) if cross_attention else None
        self.sep_token = nn.Parameter(torch.randn(D_MODEL) * 0.02)
        self.mamba_blocks = nn.ModuleList([MambaBlock(D_MODEL, MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND)
                                           for _ in range(MAMBA_NUM_BLOCKS)])
        self.pool_attn = nn.Sequential(nn.Linear(D_MODEL, POOL_HIDDEN), nn.Tanh(), nn.Linear(POOL_HIDDEN, 1))
        self.dropout = nn.Dropout(DROPOUT)
        self.classifier = nn.Linear(D_MODEL, N_STAGES)

    def pack(self, Hv, mv, Ht, mt):
        """Per sample: valid vib tokens, SEP, valid temp tokens, then padding (masks must be prefix masks)."""
        B, Lv, D = Hv.shape
        Lt = Ht.shape[1]
        nv, nt = mv.sum(dim=1), mt.sum(dim=1)
        out = Hv.new_zeros(B, Lv + 1 + Lt, D)
        bv, iv = mv.nonzero(as_tuple=True)
        out[bv, iv] = Hv[bv, iv]
        out[torch.arange(B, device=Hv.device), nv] = self.sep_token.to(Hv.dtype).unsqueeze(0).expand(B, -1)
        bt, it = mt.nonzero(as_tuple=True)
        out[bt, nv[bt] + 1 + it] = Ht[bt, it]
        valid = torch.arange(Lv + 1 + Lt, device=Hv.device).unsqueeze(0) < (nv + 1 + nt).unsqueeze(1)
        return out, valid

    def forward(self, xv, tv, mv, xt, tt, mt):
        Hv = self.htt(self.vib_proj(xv), tv, modality_idx=0)
        Ht = self.htt(self.temp_proj(xt), tt, modality_idx=1)
        diag = {}
        if self.laca is not None:
            Hv, Ht, diag = self.laca(Hv, Ht, tv, tt, mv, mt)
        seq, valid = self.pack(Hv, mv, Ht, mt)
        for block in self.mamba_blocks:
            seq = block(seq)
        a = self.pool_attn(seq).squeeze(-1).masked_fill(~valid, float("-inf"))
        w = F.softmax(a, dim=1).unsqueeze(-1)
        return self.classifier(self.dropout((seq * w).sum(dim=1))), diag


def build_fusion_model(name):
    htt_time, laca_lag, xattn = ABLATION[name]
    return FEMTOFusionMamba(htt_time, laca_lag, cross_attention=xattn)


def n_numel(params):
    return sum(p.numel() for p in params)


def param_report(model):
    total = n_numel(model.parameters())
    time_emb = n_numel(model.htt.time_embedding.parameters())
    lag = n_numel(model.laca.laca_vt.lag_bias.parameters()) + n_numel(model.laca.laca_tv.lag_bias.parameters()) \
        if model.laca is not None else 0
    laca_core = n_numel(model.laca.parameters()) - lag if model.laca is not None else 0
    active = total - (0 if model.htt_time_enabled else time_emb) - (lag if not model.laca_lag_enabled else 0)
    return {"total": total, "active": active, "time_embedding": time_emb, "lag_bias": lag, "laca_core": laca_core,
            "mamba": n_numel(model.mamba_blocks.parameters())}


PARAMS, INIT_SHA = {}, {}
for name in FUSION_MODELS:
    seed_all(0)
    m = build_fusion_model(name).to(device)
    PARAMS[name] = param_report(m)
    INIT_SHA[name] = {k: hashlib.sha1(v.detach().cpu().numpy().tobytes()).hexdigest()[:10] for k, v in m.state_dict().items()}
    _types = [type(b.mamba) for b in m.mamba_blocks]
    hook(f"real_mamba_{name}", len(_types) == MAMBA_NUM_BLOCKS and all(t is MAMBA_CLASS for t in _types),
         f"fusion {name} uses {len(_types)} x {MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__} (no fallback)")
    if m.laca is not None:
        for d in (m.laca.laca_vt, m.laca.laca_tv):
            assert d.beta == (LACA_BETA if m.laca_lag_enabled else 0.0) and n_numel(d.lag_bias.parameters()) == LAG_BIAS_PARAMS
    assert m.htt.time_enabled == m.htt_time_enabled
    del m
PARAM_TABLE = pd.DataFrame(PARAMS).T[["total", "active", "time_embedding", "lag_bias", "laca_core", "mamba"]]
print(PARAM_TABLE.to_string())

_P = PARAMS
_te, _lb = _P["C"]["time_embedding"], _P["C"]["lag_bias"]
hook("abcd_same_capacity", len({_P[n]["total"] for n in "ABCD"}) == 1
     and _P["A"]["active"] - _P["D"]["active"] == _te and _P["B"]["active"] - _P["D"]["active"] == _lb
     and _P["C"]["active"] - _P["D"]["active"] == _te + _lb,
     f"A/B/C/D built identically ({_P['D']['total']:,} params each); ACTIVE params differ from D only by the HTT time "
     f"embedding ({_te:,}: A, C) and the two LACA lag-bias nets ({_lb:,}: B, C)")
hook("abcd_same_init", all(INIT_SHA[n] == INIT_SHA["D"] for n in "ABC"),
     "A/B/C/D have byte-identical initial weights at the same seed (only the switches differ)")
hook("f0_smaller", _P["F0"]["total"] <= _P["D"]["total"] - _P["D"]["laca_core"],
     f"F0 {_P['F0']['total']:,} params vs D {_P['D']['total']:,} (no cross-attention module: -{_P['D']['laca_core']:,}, "
     f"{1 - _P['F0']['total'] / _P['D']['total']:.0%} smaller)")

# --- padding invariance: a window's logits must not depend on how much padding its batch carries ---
seed_all(0)
_m = build_fusion_model("C").to(device).eval()
with torch.no_grad():
    def _fake(nv, nt, Lv, Lt):
        xv = torch.zeros(1, Lv, N_VIB_FEAT, device=device); xt = torch.zeros(1, Lt, N_TEMP_FEAT, device=device)
        tv = torch.zeros(1, Lv, device=device); tt = torch.zeros(1, Lt, device=device)
        g = torch.Generator(device="cpu").manual_seed(7)
        xv[0, :nv] = torch.randn(nv, N_VIB_FEAT, generator=g).to(device); xt[0, :nt] = torch.randn(nt, N_TEMP_FEAT, generator=g).to(device)
        tv[0, :nv] = torch.linspace(0, 9.9, nv, device=device); tt[0, :nt] = torch.linspace(0.5, 9.5, nt, device=device)
        mv = torch.arange(Lv, device=device)[None] < nv; mt = torch.arange(Lt, device=device)[None] < nt
        return xv, tv, mv, xt, tt, mt
    _a = _m(*_fake(20, 4, 20, 4))[0]
    _b = _m(*_fake(20, 4, 75, 13))[0]
_pad_diff = float((_a - _b).abs().max())
hook("padding_invariance", _pad_diff < 1e-3, f"max |logit difference| for the same window with vs without extra "
     f"padding = {_pad_diff:.2e}", stop=False)
del _m
torch.cuda.empty_cache()

In [ ]:
# ==============================================================================
# F7. TRAINING / EVALUATION (identical procedure to the baseline Mamba runs) + resumable fusion CSV
# ==============================================================================
from scipy import stats


class FusionWindowDataset(Dataset):
    def __init__(self, idx):
        self.idx = np.asarray(idx)

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        w = int(self.idx[i])
        s = WIN["start_s"][w]
        vlo, vhi, tlo, thi = WIN["vlo"][w], WIN["vhi"][w], WIN["tlo"][w], WIN["thi"][w]
        return (VT["feat"][vlo:vhi], ((VT["tau"][vlo:vhi] - s) / TAU_UNIT_S).astype(np.float32),
                TT["feat"][tlo:thi], ((TT["tau"][tlo:thi] - s) / TAU_UNIT_S).astype(np.float32), int(WIN["label"][w]), w)


def fusion_collate(batch):
    B = len(batch)
    Lv, Lt = max(len(b[1]) for b in batch), max(len(b[3]) for b in batch)
    xv, tv, mv = torch.zeros(B, Lv, N_VIB_FEAT), torch.zeros(B, Lv), torch.zeros(B, Lv, dtype=torch.bool)
    xt, tt, mt = torch.zeros(B, Lt, N_TEMP_FEAT), torch.zeros(B, Lt), torch.zeros(B, Lt, dtype=torch.bool)
    for i, (fv, tauv, ft, taut, _, _) in enumerate(batch):
        xv[i, :len(tauv)], tv[i, :len(tauv)], mv[i, :len(tauv)] = torch.from_numpy(fv), torch.from_numpy(tauv), True
        xt[i, :len(taut)], tt[i, :len(taut)], mt[i, :len(taut)] = torch.from_numpy(ft), torch.from_numpy(taut), True
    y = torch.tensor([b[4] for b in batch], dtype=torch.long)
    w = torch.tensor([b[5] for b in batch], dtype=torch.long)
    return xv, tv, mv, xt, tt, mt, y, w


def fusion_loader(idx, shuffle, seed):
    return DataLoader(FusionWindowDataset(idx), batch_size=BATCH_SIZE if shuffle else EVAL_BATCH_SIZE, shuffle=shuffle,
                      num_workers=FUSION_NUM_WORKERS, collate_fn=fusion_collate,
                      generator=make_generator(seed) if shuffle else None)


FUSION_NORM = {}


def fusion_norm(test_c, train_bearings):
    """Per-feature mean / SD over the TRAINING bearings' tokens only (per fold, per modality)."""
    if test_c not in FUSION_NORM:
        out = []
        for T_ in (VT, TT):
            f = T_["feat"][np.isin(T_["bearing"], train_bearings)].astype(np.float64)
            out.append((torch.tensor(f.mean(0), dtype=torch.float32, device=device),
                        torch.tensor(f.std(0) + FEAT_NORM_EPS, dtype=torch.float32, device=device)))
        FUSION_NORM[test_c] = out
    return FUSION_NORM[test_c]


def to_device(batch, norm):
    (vmu, vsd), (tmu, tsd) = norm
    xv, tv, mv, xt, tt, mt, y, w = (t.to(device, non_blocking=True) for t in batch)
    xv = (xv - vmu) / vsd * mv.unsqueeze(-1)                     # padding stays exactly 0
    xt = (xt - tmu) / tsd * mt.unsqueeze(-1)
    return xv, tv, mv, xt, tt, mt, y, w


@torch.no_grad()
def fusion_predict(model, loader, norm):
    model.eval()
    P, Y, Wd, LVT, LTV = [], [], [], [], []
    for batch in loader:
        xv, tv, mv, xt, tt, mt, y, w = to_device(batch, norm)
        logits, diag = model(xv, tv, mv, xt, tt, mt)
        P.append(F.softmax(logits, dim=-1).float().cpu().numpy()); Y.append(y.cpu().numpy()); Wd.append(w.cpu().numpy())
        nanv = np.full(len(y), np.nan, np.float32)
        LVT.append(diag["lag_vt"].float().cpu().numpy() if "lag_vt" in diag else nanv)
        LTV.append(diag["lag_tv"].float().cpu().numpy() if "lag_tv" in diag else nanv)
    return tuple(np.concatenate(a) for a in (P, Y, Wd, LVT, LTV))


def fusion_tag(name, test_c, seed):
    return f"{name}_testC{test_c}_{seed}"


def fusion_probs_path(name, test_c, seed):
    return FUSION_PROBS_DIR / f"{fusion_tag(name, test_c, seed)}.npz"


def fusion_done_keys():
    if not FUSION_RESULTS_CSV.exists():
        return set()
    df = pd.read_csv(FUSION_RESULTS_CSV)
    keys = set()
    for r in df[df.config_tag == FUSION_CONFIG_TAG].itertuples():
        p = fusion_probs_path(r.model, int(r.test_condition), int(r.seed))
        if p.exists():
            with np.load(p, allow_pickle=False) as z:
                if str(z["config_tag"]) == FUSION_CONFIG_TAG:
                    keys.add((r.model, int(r.test_condition), int(r.seed)))
    return keys


def fusion_append(row):
    new = pd.DataFrame([row])
    if FUSION_RESULTS_CSV.exists():
        old = pd.read_csv(FUSION_RESULTS_CSV)
        stale = ((old.config_tag == row["config_tag"]) & (old.model == row["model"])
                 & (old.test_condition == row["test_condition"]) & (old.seed == row["seed"]))
        new = pd.concat([old[~stale], new], ignore_index=True)
    tmp = FUSION_RESULTS_CSV.with_suffix(".tmp")
    new.to_csv(tmp, index=False)
    tmp.replace(FUSION_RESULTS_CSV)


def fusion_run_one(name, test_c, seed):
    t0 = time.time()
    fold = FOLDS[("temperature", test_c)]                        # baseline's fold + inner-val bearings, unchanged
    idx = {s: np.where(np.isin(WIN["bearing"], fold[s]))[0] for s in ("train", "val", "test")}
    norm = fusion_norm(test_c, fold["train"])

    seed_all(seed)                                               # at the START of every run, before the model is built
    model = build_fusion_model(name).to(device)
    assert all(type(b.mamba) is MAMBA_CLASS for b in model.mamba_blocks)
    init_sha = hashlib.sha1(b"".join(v.detach().cpu().numpy().tobytes() for v in model.state_dict().values())).hexdigest()[:12]
    counts = np.bincount(WIN["label"][idx["train"]], minlength=N_STAGES)
    assert (counts > 0).all(), f"a stage is missing from training: {counts}"
    cw = torch.tensor(len(idx["train"]) / (N_STAGES * counts), dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=cw)
    optimizer = torch.optim.AdamW(model.parameters(), lr=FUSION_LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
    tr, va, te = fusion_loader(idx["train"], True, seed), fusion_loader(idx["val"], False, seed), fusion_loader(idx["test"], False, seed)
    tag = f"[fusion {name} test=C{test_c} seed {seed}]"
    print(f"{tag} init {init_sha} | {PARAMS[name]['total']:,} params ({PARAMS[name]['active']:,} active) | bearings "
          f"train {len(fold['train'])} / val {len(fold['val'])} {fold['val']} / test {len(fold['test'])} | windows "
          f"train {len(idx['train'])} / val {len(idx['val'])} / test {len(idx['test'])}")

    best_f1, best_state, best_epoch, epochs_run = -1.0, None, None, 0
    for epoch in range(1, EPOCHS + 1):
        te0 = time.time()
        model.train()
        tot, nb = 0.0, 0
        for batch in tr:
            xv, tv, mv, xt, tt, mt, y, _ = to_device(batch, norm)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xv, tv, mv, xt, tt, mt)[0], y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            tot += loss.item()
            nb += 1
        scheduler.step()
        epochs_run = epoch
        if not math.isfinite(tot):
            raise RuntimeError(f"{tag} non-finite training loss at epoch {epoch}")
        vP, vy, _, _, _ = fusion_predict(model, va, norm)
        vf1 = compute_metrics(vy, vP)["macro_f1"]
        if vf1 > best_f1:
            best_f1, best_state, best_epoch = vf1, copy.deepcopy(model.state_dict()), epoch
        print(f"  epoch {epoch:2d} | loss {tot / nb:.4f} | val macro-F1 {vf1:.4f}{' *' if epoch == best_epoch else ''} "
              f"| {time.time() - te0:.1f} s")
        if epoch - best_epoch >= EARLY_STOP_PATIENCE:
            print(f"  early stop at epoch {epoch}, no improvement for {EARLY_STOP_PATIENCE} epochs")
            break

    model.load_state_dict(best_state)
    P, y, wid, lag_vt, lag_tv = fusion_predict(model, te, norm)
    assert np.array_equal(wid, idx["test"]) and np.array_equal(y, WIN["label"][wid])
    met = compute_metrics(y, P)
    np.savez(fusion_probs_path(name, test_c, seed),
             window_id=wid, bearing=WIN["bearing"][wid], condition=WIN["condition"][wid], stage_label=y,
             probs=P.astype(np.float32), pred=P.argmax(1), centre_s=WIN["centre_s"][wid], frac_of_life=WIN["frac"][wid],
             n_vib_tokens=WIN["n_vib"][wid], n_temp_tokens=WIN["n_temp"][wid], expected_lag_vt_min=lag_vt,
             expected_lag_tv_min=lag_tv, config_tag=np.array(FUSION_CONFIG_TAG), model=np.array(name),
             test_condition=np.int64(test_c), seed=np.int64(seed))
    torch.save({"state_dict": best_state, "best_epoch": best_epoch, "best_val_macro_f1": best_f1, "fold": fold,
                "norm": [(a.cpu().numpy(), b.cpu().numpy()) for a, b in norm], "config_tag": FUSION_CONFIG_TAG,
                "model": name, "ablation": ABLATION[name], "test_condition": test_c, "seed": seed},
               FUSION_CKPT_DIR / f"{fusion_tag(name, test_c, seed)}.pt")
    row = {"config_tag": FUSION_CONFIG_TAG, "model": name, "htt_time": ABLATION[name][0], "laca_lag": ABLATION[name][1],
           "cross_attention": ABLATION[name][2], "test_condition": test_c, "seed": seed,
           "n_bearings_fusion": N_FUSION_BEARINGS, "n_train_bearings": len(fold["train"]),
           "n_val_bearings": len(fold["val"]), "n_test_bearings": len(fold["test"]),
           "train_bearings": " ".join(fold["train"]), "val_bearings": " ".join(fold["val"]),
           "test_bearings": " ".join(fold["test"]), "n_train_windows": len(idx["train"]),
           "n_val_windows": len(idx["val"]), "n_test_windows": len(idx["test"]),
           "n_params_total": PARAMS[name]["total"], "n_params_active": PARAMS[name]["active"], "init_sha": init_sha,
           "best_epoch": best_epoch, "epochs_run": epochs_run, "best_val_macro_f1": best_f1,
           **{k: v for k, v in met.items() if k != "confusion_matrix"},
           "confusion_matrix": json.dumps(met["confusion_matrix"].tolist()),
           "mean_expected_lag_vt_min": float(np.nanmean(lag_vt)) if np.isfinite(lag_vt).any() else np.nan,
           "mean_expected_lag_tv_min": float(np.nanmean(lag_tv)) if np.isfinite(lag_tv).any() else np.nan,
           "runtime_s": round(time.time() - t0, 1), "finished_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    fusion_append(row)
    print(f"{tag} TEST: acc {met['accuracy']:.4f} | macro-F1 {met['macro_f1']:.4f} | bal-acc "
          f"{met['balanced_accuracy']:.4f} | recall " + " ".join(f"{n} {met['recall_' + n]:.3f}" for n in STAGE_NAMES)
          + f" | best epoch {best_epoch}/{epochs_run} | {time.time() - t0:.0f} s")
    del model, best_state
    torch.cuda.empty_cache()

In [ ]:
# ==============================================================================
# F8. RUN ALL (gate on the LATEST result of every hard hook; resumable under FUSION_CONFIG_TAG)
# ==============================================================================
_hard_fail = [n for n, v in HOOKS.items() if (not v[-1][0]) and v[-1][2]]
if _hard_fail:
    raise RuntimeError(f"Hard verification hooks failed: {_hard_fail} — not training.")

FUSION_T0 = time.time()
_done = fusion_done_keys()
_valid_c = [c for c in FOLD_CONDITIONS if FOLDS[("temperature", c)]["valid"]]
print(f"{len(FUSION_MODELS) * len(SEEDS) * len(_valid_c)} fusion runs; {len(_done)} already complete under "
      f"{FUSION_CONFIG_TAG}; folds {_valid_c}")
for c in FOLD_CONDITIONS:
    if c not in _valid_c:
        print(f"SKIP fold test=C{c}: {FOLDS[('temperature', c)]['reason']}")
        continue
    for s in SEEDS:
        for name in FUSION_MODELS:
            if (name, c, s) in _done:
                print(f"skip (already logged) {fusion_tag(name, c, s)}")
                continue
            fusion_run_one(name, c, s)
print(f"Fusion training finished in {(time.time() - FUSION_T0) / 60:.1f} min (this session)")

## Results

**Per condition** is the fold that tests on that operating condition, as mean ± SD over seeds. **Pooled** means: for
each seed, concatenate the test windows of all folds (every one of the 13 bearings is tested exactly once), compute
the metrics, then take mean ± SD over seeds. The metrics are the baseline's own `compute_metrics`.

In [ ]:
# ==============================================================================
# F9. RESULTS TABLES — A/B/C/D/F0, pooled and per condition
# ==============================================================================
FRES = pd.read_csv(FUSION_RESULTS_CSV)
FRES = FRES[FRES.config_tag == FUSION_CONFIG_TAG].copy()
METRICS = ["accuracy", "macro_f1", "balanced_accuracy"] + [f"recall_{n}" for n in STAGE_NAMES]


def pm(x):
    x = np.asarray(x, float)
    return f"{x.mean():.4f} ± {x.std(ddof=1):.4f}" if len(x) > 1 else (f"{x[0]:.4f}" if len(x) else "-")


def pooled_metrics(paths):
    ys, Ps, bs = [], [], []
    for p in paths:
        with np.load(p, allow_pickle=False) as z:
            ys.append(z["stage_label"]); Ps.append(z["probs"]); bs.append(z["bearing"])
    met = compute_metrics(np.concatenate(ys), np.concatenate(Ps))
    met["n_bearings"] = len(set(np.concatenate(bs).tolist()))
    return met


_rows, FUSION_POOLED_CM = [], {}
for name in FUSION_MODELS:
    per_seed = []
    for s in SEEDS:
        paths = [fusion_probs_path(name, c, s) for c in _valid_c]
        if all(p.exists() for p in paths):
            per_seed.append(pooled_metrics(paths))
    if per_seed:
        FUSION_POOLED_CM[name] = np.mean([d["confusion_matrix"] for d in per_seed], axis=0)
        _rows.append({"model": name, "level": f"POOLED ({len(_valid_c)} folds)",
                      "bearings": f"{per_seed[0]['n_bearings']} of {N_FUSION_BEARINGS}", "seeds": len(per_seed),
                      **{k: pm([d[k] for d in per_seed]) for k in METRICS}})
for (name, c), g in FRES.groupby(["model", "test_condition"], sort=True):
    _rows.append({"model": name, "level": f"test C{c} ({CONDITIONS[c]})",
                  "bearings": f"{int(g.n_test_bearings.iloc[0])} of {N_FUSION_BEARINGS} ({g.test_bearings.iloc[0]})",
                  "seeds": len(g), **{k: pm(g[k]) for k in METRICS}})
FUSION_TABLE = pd.DataFrame(_rows)
print(f"Chance balanced accuracy = {1 / N_STAGES:.3f}. Fusion bearings: {N_FUSION_BEARINGS} of {N_FUSION_BEARINGS} "
      f"(both modalities required). Seeds {SEEDS}. Window {WINDOW_MINUTES} min / stride {WINDOW_STRIDE_MINUTES} min.\n")
with pd.option_context("display.width", 300, "display.max_columns", 30, "display.max_colwidth", 70):
    print(FUSION_TABLE.to_string(index=False))
for name, cm in FUSION_POOLED_CM.items():
    print(f"\nPooled confusion matrix — {name} ({N_FUSION_BEARINGS} bearings), mean over seeds; rows true, cols predicted")
    print(pd.DataFrame(np.round(cm / cm.sum(1, keepdims=True), 3), index=[f"true {n}" for n in STAGE_NAMES],
                       columns=[f"pred {n}" for n in STAGE_NAMES]).to_string())
_lag = FRES.groupby("model")[["mean_expected_lag_vt_min", "mean_expected_lag_tv_min"]].mean()
print("\nMean attention-weighted lag (key time − query time, minutes; LACA models only):")
print(_lag.round(3).to_string())

## PROB-AVG reference (analysis only, NOT a trained model)

PROB-AVG averages the **baseline notebook's saved test probabilities** of its two single-modality Mamba models
(vibration-mamba and temperature-mamba), for the same test condition and seed. No training happens here. It needs the
baseline's `femto_probs/*.npz`: either this session's `/kaggle/working/femto_probs`, or the baseline notebook's
output attached as a Kaggle input (any `/kaggle/input/**/femto_probs`). If they are not found, this section is
skipped.

**Caveat — not a clean paired comparison.** The baseline models predicted **single bursts / single temperature
records**, not this notebook's 10-minute windows. Here each fusion test window takes the mean of the baseline
vibration probabilities over the bursts inside it and the mean of the temperature probabilities over the records
inside it, and then averages the two. The result is scored against this notebook's window labels. Several things
differ from the fused models:
- the windowing scheme and the label each base model was trained on (per burst, not per window centre);
- the vibration baseline's training folds also contained the 4 bearings without temperature;
- the baseline's temperature windows dropped records shorter than 600 rows.

So a gap between PROB-AVG and A/B/C/D/F0 mixes the effect of fusion with all of these differences. Treat it as a
rough reference only.

In [ ]:
# ==============================================================================
# F10. PROB-AVG — average of the baseline's saved single-modality Mamba probabilities, mapped onto fusion windows
# ==============================================================================
import urllib.parse

BASELINE_PROB_NAMES = [f"{run_tag(m, 'mamba', c, s)}.npz" for m in ("vibration", "temperature")
                       for c in FOLD_CONDITIONS for s in SEEDS]
BASELINE_PROB_DIR_NAMES = ["femto_probs", "Baseline Probs"]       # exact names first, then case-insensitive


def _n_required(d):
    return sum((d / n).exists() for n in BASELINE_PROB_NAMES)


def _find_probs_folder(root):
    """Inside a cloned repo: a folder named exactly femto_probs / Baseline Probs, else a case-insensitive match."""
    dirs = [Path(r) for r, _, _ in os.walk(root) if ".git" not in Path(r).parts]
    for exact in (True, False):
        for d in dirs:
            if any(d.name == n if exact else d.name.lower() == n.lower() for n in BASELINE_PROB_DIR_NAMES) and _n_required(d):
                return d
    return None


def _download_expected(base_url, dest):
    """A folder URL cannot be listed, so fetch the expected baseline file names from it one by one."""
    dest.mkdir(parents=True, exist_ok=True)
    got = 0
    for n in BASELINE_PROB_NAMES:
        if (dest / n).exists():
            got += 1
            continue
        try:
            req = urllib.request.Request(f"{base_url.rstrip('/')}/{urllib.parse.quote(n)}", headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120) as r, open(dest / n, "wb") as f:
                f.write(r.read())
            got += 1
        except Exception as exc:
            print(f"    {n}: not downloaded ({exc.__class__.__name__}: {exc})")
    return dest if got else None


def fetch_baseline_probs_from_github(url):
    url = url.strip().rstrip("/")
    dest = SCRATCH_DIR / "baseline_probs_github"
    m_tree = re.match(r"^https?://github\.com/([^/]+)/([^/]+)/tree/([^/]+)/(.+)$", url)
    m_repo = re.match(r"^https?://github\.com/([^/]+)/([^/]+?)(?:\.git)?$", url)
    if m_tree:                                                    # folder inside a repo -> its raw-content base URL
        o, r, branch, path = m_tree.groups()
        path = "/".join(urllib.parse.quote(urllib.parse.unquote(p)) for p in path.split("/"))
        base = f"https://raw.githubusercontent.com/{o}/{r}/{branch}/{path}"
        print(f"  GitHub folder URL -> downloading expected files from {base}")
        return _download_expected(base, dest / "folder")
    if m_repo:                                                    # whole repo -> shallow clone, then find the folder
        clone = dest / f"{m_repo.group(1)}__{m_repo.group(2)}"
        if not (clone / ".git").exists():
            print(f"  git clone --depth 1 {url} (the whole repo is downloaded; a /tree/<branch>/<folder> URL fetches less)")
            r = subprocess.run(["git", "clone", "--depth", "1", url, str(clone)], capture_output=True, text=True)
            if r.returncode != 0:
                print(f"    clone failed (private repo or no internet?): {(r.stderr or r.stdout)[-500:]}")
                return None
        return _find_probs_folder(clone)
    print(f"  direct / raw folder URL -> downloading expected files from {url}")
    return _download_expected(url, dest / "direct")


def find_baseline_probs_dir():
    """(a) this session's /kaggle/working/femto_probs; (b) an attached Kaggle input; (c) BASELINE_PROBS_GITHUB_URL."""
    if _n_required(PROBS_DIR):
        return PROBS_DIR, "(a) same-session output"
    if Path("/kaggle/input").is_dir():
        cands = (sorted(Path("/kaggle/input").glob("*/femto_probs")) + sorted(Path("/kaggle/input").glob("*"))
                 + sorted(Path("/kaggle/input").glob("**/femto_probs")))
        d = next((d for d in cands if d.is_dir() and _n_required(d)), None)
        if d is not None:
            return d, "(b) attached Kaggle input"
    if BASELINE_PROBS_GITHUB_URL.strip():
        print(f"(a) and (b) found nothing; trying (c) BASELINE_PROBS_GITHUB_URL = {BASELINE_PROBS_GITHUB_URL}")
        d = fetch_baseline_probs_from_github(BASELINE_PROBS_GITHUB_URL)
        if d is not None and _n_required(d):
            return d, f"(c) GitHub ({BASELINE_PROBS_GITHUB_URL})"
        print("  (c) GitHub fallback found no baseline probability files.")
    return None, None


BASE_PROBS_DIR, BASE_PROBS_SOURCE = find_baseline_probs_dir()
PROBAVG_ROWS, PROBAVG_PER = [], {}
if BASE_PROBS_DIR is None:
    print("PROB-AVG SKIPPED: the baseline's femto_probs/*.npz were not found in /kaggle/working/femto_probs, under "
          "/kaggle/input, or via BASELINE_PROBS_GITHUB_URL" + (" (not set)" if not BASELINE_PROBS_GITHUB_URL.strip() else "")
          + ". Attach the baseline notebook's output as an input, or set BASELINE_PROBS_GITHUB_URL, to enable it.")
else:
    print(f"PROB-AVG source used: {BASE_PROBS_SOURCE} -> {BASE_PROBS_DIR}")
    print(f"  {len(list(BASE_PROBS_DIR.glob('*.npz')))} .npz files in that folder; {_n_required(BASE_PROBS_DIR)} of the "
          f"{len(BASELINE_PROB_NAMES)} needed (vibration-mamba + temperature-mamba x {len(FOLD_CONDITIONS)} folds x "
          f"{len(SEEDS)} seeds)")
    _tags = set()
    for c in _valid_c:
        for s in SEEDS:
            pv, pt = (BASE_PROBS_DIR / f"{run_tag(m, 'mamba', c, s)}.npz" for m in ("vibration", "temperature"))
            if not (pv.exists() and pt.exists()):
                print(f"  missing baseline predictions for test=C{c} seed {s} — skipped")
                continue
            maps = []
            for p in (pv, pt):
                with np.load(p, allow_pickle=False) as z:
                    _tags.add(str(z["config_tag"]))
                    maps.append({(b, int(n)): pr for b, n, pr in zip(z["bearing"], z["file_num"], z["probs"])})
            wids = np.where(np.isin(WIN["bearing"], FOLDS[("temperature", c)]["test"]))[0]
            P, n_vib_only = [], 0
            for w in wids:
                vp = [maps[0].get((VT["bearing"][i], int(VT["num"][i]))) for i in range(WIN["vlo"][w], WIN["vhi"][w])]
                tp = [maps[1].get((TT["bearing"][i], int(TT["num"][i]))) for i in range(WIN["tlo"][w], WIN["thi"][w])]
                vp, tp = [x for x in vp if x is not None], [x for x in tp if x is not None]
                assert vp, f"no baseline vibration predictions inside window {w}"
                if tp:
                    P.append(0.5 * (np.mean(vp, axis=0) + np.mean(tp, axis=0)))
                else:
                    P.append(np.mean(vp, axis=0)); n_vib_only += 1
            P = np.array(P)
            met = compute_metrics(WIN["label"][wids], P)
            PROBAVG_PER[(c, s)] = (WIN["label"][wids], P, WIN["bearing"][wids])
            PROBAVG_ROWS.append({"test_condition": c, "seed": s, "n_test_bearings": len(set(WIN["bearing"][wids])),
                                 "windows": len(wids), "vib_only_windows": n_vib_only,
                                 **{k: v for k, v in met.items() if k != "confusion_matrix"}})
    print(f"  baseline config tag(s) found: {sorted(_tags)} (this session's baseline constants: {BASELINE_CONFIG_TAG})")
    if PROBAVG_ROWS:
        _pa = pd.DataFrame(PROBAVG_ROWS)
        _rows = []
        _pooled = []
        for s in SEEDS:
            keys = [(c, s) for c in _valid_c if (c, s) in PROBAVG_PER]
            if len(keys) == len(_valid_c):
                y = np.concatenate([PROBAVG_PER[k][0] for k in keys]); P = np.concatenate([PROBAVG_PER[k][1] for k in keys])
                _pooled.append(compute_metrics(y, P))
        if _pooled:
            _rows.append({"model": "PROB-AVG (analysis)", "level": f"POOLED ({len(_valid_c)} folds)",
                          "bearings": f"{N_FUSION_BEARINGS} of {N_FUSION_BEARINGS}", "seeds": len(_pooled),
                          **{k: pm([d[k] for d in _pooled]) for k in METRICS}})
        for c, g in _pa.groupby("test_condition"):
            _rows.append({"model": "PROB-AVG (analysis)", "level": f"test C{c} ({CONDITIONS[c]})",
                          "bearings": f"{int(g.n_test_bearings.iloc[0])} of {N_FUSION_BEARINGS}", "seeds": len(g),
                          **{k: pm(g[k]) for k in METRICS}})
        FUSION_TABLE = pd.concat([FUSION_TABLE, pd.DataFrame(_rows)], ignore_index=True)
        with pd.option_context("display.width", 300, "display.max_columns", 30):
            print(pd.DataFrame(_rows).to_string(index=False))
        print("CAVEAT: PROB-AVG is an analysis of the baseline's per-burst / per-record predictions mapped onto these "
              "windows — different windowing, training labels and training bearings. Not a clean paired comparison.")
FUSION_TABLE.to_csv(FUSION_SUMMARY_CSV, index=False)

## Paired ablation comparisons and verdict

Every comparison pairs two models on the same (test condition, seed). That means the same initial weights for
A/B/C/D, the same batches and the same test windows. Primary metric: test macro-F1. Each comparison reports the mean
and SD of the differences, wins/ties/losses, and a paired t-test over the 6 pairs. The pooled-per-seed differences are
also shown. The verdict applies the pre-registered rule from the top of the notebook.

In [ ]:
# ==============================================================================
# F11. PAIRED COMPARISONS (mean / SD / wins / paired t) + VERDICT
# ==============================================================================
COMPARISONS = [("B", "D", "LACA lag bias, HTT off"), ("C", "A", "LACA lag bias, HTT on"),
               ("A", "D", "HTT time embedding, lag bias off"), ("C", "B", "HTT time embedding, lag bias on"),
               ("C", "D", "full mechanism vs no time information"), ("D", "F0", "content-only cross-attention vs none"),
               ("A", "B", "time in tokens vs time in the attention bias")]
_piv = FRES.pivot_table(index=["test_condition", "seed"], columns="model", values="macro_f1")
_pooled_f1 = {}
for name in FUSION_MODELS:
    for s in SEEDS:
        paths = [fusion_probs_path(name, c, s) for c in _valid_c]
        if all(p.exists() for p in paths):
            _pooled_f1[(name, s)] = pooled_metrics(paths)["macro_f1"]


def paired(a, b):
    if a not in _piv or b not in _piv:
        return None
    d = (_piv[a] - _piv[b]).dropna()
    n = len(d)
    t, p = (stats.ttest_rel(_piv.loc[d.index, a], _piv.loc[d.index, b]) if n >= 2 and d.std(ddof=1) > 0
            else (np.nan, np.nan))
    pooled = [_pooled_f1[(a, s)] - _pooled_f1[(b, s)] for s in SEEDS if (a, s) in _pooled_f1 and (b, s) in _pooled_f1]
    return {"n_pairs": n, "mean_diff": d.mean(), "sd_diff": d.std(ddof=1) if n > 1 else np.nan,
            "wins": int((d > 0).sum()), "ties": int((d == 0).sum()), "losses": int((d < 0).sum()),
            "t": float(t), "p": float(p), "pooled_diff_per_seed": " ".join(f"{x:+.4f}" for x in pooled), "diffs": d}


_prow = []
PAIRED = {}
for a, b, what in COMPARISONS:
    r = paired(a, b)
    if r is None:
        continue
    PAIRED[(a, b)] = r
    _prow.append({"comparison": f"{a} − {b}", "isolates": what, **{k: v for k, v in r.items() if k != "diffs"}})
PAIRED_TABLE = pd.DataFrame(_prow)
with pd.option_context("display.width", 300, "display.max_columns", 30):
    print(f"Test macro-F1 differences, paired by (test condition, seed); {N_FUSION_BEARINGS} of {N_FUSION_BEARINGS} bearings")
    print(PAIRED_TABLE.round(4).to_string(index=False))
    print("\nPer-pair macro-F1 (rows = test condition, seed):")
    print(_piv.round(4).to_string())
PAIRED_TABLE.to_csv(FUSION_PAIRED_CSV, index=False)

# ---------------- verdict (pre-registered rule) ----------------
_n_expected = len(_valid_c) * len(SEEDS)
_lag = [PAIRED.get(("B", "D")), PAIRED.get(("C", "A"))]
if any(r is None or r["n_pairs"] < _n_expected for r in _lag):
    VERDICT = (f"VERDICT PENDING: the lag-bias comparisons need all {_n_expected} (fold, seed) pairs for B, C, A and D; "
               f"have {[r['n_pairs'] if r else 0 for r in _lag]}.")
else:
    def _passes(r):
        return r["mean_diff"] > 0 and r["wins"] >= VERDICT_MIN_WINS and r["p"] < VERDICT_ALPHA

    def _fmt(k):
        r = PAIRED[k]
        return (f"{k[0]} − {k[1]} = {r['mean_diff']:+.4f} ± {r['sd_diff']:.4f} macro-F1, {r['wins']}/{r['n_pairs']} "
                f"wins, paired t = {r['t']:.2f}, p = {r['p']:.3f}")
    _yes = all(_passes(r) for r in _lag)
    _htt = PAIRED.get(("A", "D")); _xa = PAIRED.get(("D", "F0"))
    VERDICT = (
        f"VERDICT — does LACA's lag bias give a reliable benefit at the corrected, trajectory scale "
        f"({WINDOW_MINUTES}-min windows, burst tokens, anchor {ANCHOR_MODE!r})? {'YES' if _yes else 'NO'}. "
        f"Lag bias without the HTT time embedding: {_fmt(('B', 'D'))}. Lag bias on top of HTT: {_fmt(('C', 'A'))}. "
        f"The pre-registered rule needs both to have a positive mean, >= {VERDICT_MIN_WINS}/{_n_expected} wins and "
        f"p < {VERDICT_ALPHA}; {'both pass' if _yes else 'at least one fails'}. "
        + (f"For context, the HTT time embedding alone gives {_fmt(('A', 'D'))}, and content-only cross-attention "
           f"vs plain concatenation gives {_fmt(('D', 'F0'))}. " if _htt and _xa else "")
        + f"These are {_n_expected} paired runs ({len(_valid_c)} folds x {len(SEEDS)} seeds) on {N_FUSION_BEARINGS} "
        f"bearings, a search-phase sample: "
        + ("the benefit is consistent enough to justify a confirmatory run with more seeds before it is claimed."
           if _yes else "a NO means no reliable benefit was shown here, not that one is ruled out; nothing in these "
           "numbers supports claiming the lag bias helps.")
    )
print("\n" + VERDICT)

In [ ]:
# ==============================================================================
# F12. DELIVERABLES SUMMARY — hooks, anchor table, token density, parameters, runtime
# ==============================================================================
print("VERIFICATION HOOKS (latest result of each)")
for name, entries in HOOKS.items():
    ok, msg, stop = entries[-1]
    print(f"  {'PASS' if ok else 'FAIL'}{'' if stop else ' (soft)'} | {name}: {msg}")
print("\nANCHOR CORRECTION (STEP 2)")
print(ANCHOR_TABLE.to_string(index=False))
print("\nTOKEN DENSITY (STEP 3)")
print(DENSITY_TABLE.to_string(index=False))
print("\nPARAMETERS")
print(PARAM_TABLE.to_string())
print(f"\nBearings: fusion {N_FUSION_BEARINGS} of {N_FUSION_BEARINGS} (both modalities); zero-temperature list from the "
      f"baseline: {sorted(EXPECTED_NO_TEMP)}")
print(f"Runs logged under {FUSION_CONFIG_TAG}: {len(FRES)} | summed training time {FRES.runtime_s.sum() / 60:.1f} min")
print(f"Total notebook wall time this session: {(time.time() - NOTEBOOK_T0) / 60:.1f} min")
print(f"Outputs: {FUSION_RESULTS_CSV}, {FUSION_SUMMARY_CSV}, {FUSION_PAIRED_CSV}, {FUSION_PROBS_DIR}/, {FUSION_CKPT_DIR}/")
print("\n" + VERDICT)

In [ ]:
# ==============================================================================
# F13. REMINDER — persist this notebook's outputs
# ==============================================================================
print("=" * 90)
print("=" * 90)
print("REMINDER: click 'Save Version' -> 'Save & Run All' now if you want these results,")
print("checkpoints and probability files (under /kaggle/working/) to persist. Everything")
print("in /kaggle/working/ is lost when this session ends unless you do this.")
print(f"  results: {FUSION_RESULTS_CSV.name}, {FUSION_SUMMARY_CSV.name}, {FUSION_PAIRED_CSV.name}")
print(f"  folders: {FUSION_PROBS_DIR.name}/, {FUSION_CKPT_DIR.name}/")
print("=" * 90)
print("=" * 90)